In [ ]:
# %%
# ==================================================================================================
# POOLED A2D2-COEFFICIENT EEG VARIANCE TOPOMAP CLASSIFICATION
# MEDGEMMA-4B-IT + LoReFT R4
#
# Train/validation:
#   - Pooled A2D2 training CSV from Patients 08, 11, 18, and 24
#   - 2875 total topomaps
#   - Deterministic stratified 80% train / 20% validation split
#
# Final testing:
#   - Reload the best LoReFT checkpoint into a fresh MedGemma model
#   - Evaluate separately on blinded A2D2 test CSVs for Patients 08, 11, 18, and 24
#
# LoReFT settings follow the reference script:
#   - rank = 4
#   - language layers = [8, 17, 26, 31]
#   - prefix prompt tokens = 5
#   - suffix prompt tokens = 5
#   - dropout = 0.0
#   - LR = 9e-4
#   - epoch limit = 50
#   - early stopping patience = 7 on validation balanced accuracy
#   - gradient checkpointing OFF
#   - LoReFT parameters and LoReFT computation stay in true FP32
#
# Labels:
#   0 = non-seizure
#   1 = seizure
# ==================================================================================================

import gc
import json
import math
import os
import random
import re
import sys
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
)
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import transformers
from huggingface_hub import whoami
from transformers import AutoModelForMultimodalLM, AutoProcessor, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")


# %%
# ==================================================================================================
# 1. PATHS
# ==================================================================================================

TRAINING_METADATA_CSV = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\A2D2\Training\training_metadata.csv"
)

TEST_METADATA_CSVS = {
    "08": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patient 08\A2D2\Testing\Blinnded"
        r"\Patient_08_A2D2_blinded_test_labels.csv"
    ),
    "11": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patient 11\A2D2\TRAINING_TESTING_SPLIT_seed_99"
        r"\Testing\Blinded Topomaps\patient11_A2D2_blinded_labels.csv"
    ),
    "18": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patient 18\A2D2\Testing\Blinnded"
        r"\Patient_18_A2D2_blinded_test_labels.csv"
    ),
    "24": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patiient 24\A2D2\Testing\Blinnded"
        r"\Patient_24_A2D2_blinded_test_labels.csv"
    ),
}

# Keep this output path short because OneDrive + deep checkpoint folders can hit Windows path limits.
OUTPUT_BASE = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4"
)

RUN_ID = time.strftime("%y%m%d_%H%M%S")
OUTPUT_ROOT = os.path.join(OUTPUT_BASE, RUN_ID)
BEST_CHECKPOINT_ROOT = os.path.join(OUTPUT_ROOT, "ckpt")
TEST_OUTPUT_ROOT = os.path.join(OUTPUT_ROOT, "blinded_tests")

os.makedirs(BEST_CHECKPOINT_ROOT, exist_ok=True)
os.makedirs(TEST_OUTPUT_ROOT, exist_ok=True)

TRAIN_SPLIT_CSV = os.path.join(OUTPUT_ROOT, "train.csv")
VALIDATION_SPLIT_CSV = os.path.join(OUTPUT_ROOT, "val.csv")
TRAINING_HISTORY_CSV = os.path.join(OUTPUT_ROOT, "history.csv")
EXPERIMENT_CONFIG_JSON = os.path.join(OUTPUT_ROOT, "config.json")
TARGET_MODULES_TXT = os.path.join(OUTPUT_ROOT, "target.txt")
ALL_PATIENT_METRICS_CSV = os.path.join(OUTPUT_ROOT, "all_patient_metrics.csv")
ALL_PATIENT_METRICS_JSON = os.path.join(OUTPUT_ROOT, "all_patient_metrics.json")

BEST_ADAPTER_DIR = None
BEST_REFT_WEIGHTS = None
BEST_REFT_CONFIG = None


# %%
# ==================================================================================================
# 2. MODEL, PROMPTS, RANDOM SEED
# ==================================================================================================

MODEL_ID = "google/medgemma-4b-it"

SYSTEM_PROMPT = """
You are a strict vision-language classifier for EEG seizure detection.
Classify each EEG topomap image into exactly one of two labels:
Seizure
Non-seizure
Return only the final label. Do not provide explanation, punctuation, confidence score, or extra text.
""".strip()

USER_PROMPT = """
You are an expert in classifying seizure and non-seizure EEG patterns from composite wavelet-coefficient variance topomaps.
The attached composite image represents one 2-second EEG window:
- Left panel: A2 approximation-coefficient variance topomap, representing approximately 0-32 Hz.
- Right panel: D2 detail-coefficient variance topomap, representing approximately 32-64 Hz.
The EEG was divided into non-overlapping 2-second windows, filtered using a fourth-order Butterworth 59-61 Hz band-stop filter, and decomposed using a level-2 Discrete Wavelet Transform. For each bipolar EEG channel, the A2 and D2 coefficient variances were calculated and spatially interpolated at the midpoint between the two electrodes to create the topomaps. Each panel uses its own fixed coefficient-specific color scale across the complete dataset. Therefore, evaluate the relative intensity and spatial distribution within each panel. Do not directly compare the same color between the A2 and D2 panels as the same numerical value. Yellow indicates relatively higher coefficient variance within the corresponding panel, while dark purple indicates relatively lower coefficient variance.
Internally evaluate the image in three steps:
1. Examine the A2 panel for the intensity, spatial extent, localization, symmetry, and organization of elevated variance.
2. Examine the D2 panel independently using the same characteristics.
3. Combine the evidence from both panels and classify any image as seizure if both of the topomaps in one image look like seizure.
Return only:
seizure
non-seizure
""".strip()

RANDOM_SEED = 42


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_all_seeds(RANDOM_SEED)


# %%
# ==================================================================================================
# 3. LoReFT + TRAINING SETTINGS
# ==================================================================================================

LOREFT_RANK = 4
LOREFT_TARGET_LAYERS_HUMAN = [8, 17, 26, 31]
NUM_LOREFT_INTERVENTIONS = len(LOREFT_TARGET_LAYERS_HUMAN)
LOREFT_PREFIX_TOKENS = 5
LOREFT_SUFFIX_TOKENS = 5
LOREFT_DROPOUT = 0.0
LOREFT_COMPONENT = "language transformer block output / residual representation"
LOREFT_MODULE_NAME = "_loreft_intervention"

NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 7
LEARNING_RATE = 9e-4
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 1
VALIDATION_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.05
MAX_NEW_TOKENS = 8
NUM_WORKERS = 0

EXPECTED_FULL_TRAIN_TOTAL = 2875
EXPECTED_ACTUAL_TRAIN_TOTAL = 2300
EXPECTED_VALIDATION_TOTAL = 575
TRAIN_FRACTION = 0.80


# %%
# ==================================================================================================
# 4. ENVIRONMENT CHECKS
# ==================================================================================================

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "=" * 100)
print("HUGGING FACE LOGIN CHECK")
print("=" * 100)

try:
    hf_info = whoami()
    print("\nLogged in as:", hf_info["name"])
except Exception as error:
    raise RuntimeError(
        "\nHugging Face authentication is not available.\n"
        "Run this once in a notebook cell first:\n"
        "from huggingface_hub import login\n"
        "login()"
    ) from error

print("\n" + "=" * 100)
print("GPU CHECK")
print("=" * 100)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

DEVICE = torch.device("cuda:0")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"\nGPU: {GPU_NAME}")
print(f"GPU memory: {GPU_MEMORY_GB:.2f} GB")

if torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
    PRECISION_NAME = "bfloat16"
else:
    MODEL_DTYPE = torch.float16
    PRECISION_NAME = "float16"

torch.backends.cuda.matmul.allow_tf32 = True

print(f"Model/training dtype: {PRECISION_NAME}")
print("\n" + "=" * 100)
print("PACKAGE VERSIONS")
print("=" * 100)
print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")


# %%
# ==================================================================================================
# 5. METADATA LOADING
# ==================================================================================================

def clean_path(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().strip('"').strip("'")
    value = os.path.expandvars(value)
    value = os.path.expanduser(value)
    return os.path.normpath(value)


def csv_path_spelling_variants(csv_path):
    candidates = [csv_path]

    replacements = [
        ("Patiient 24", "Patient 24"),
        ("Patient 24", "Patiient 24"),
        ("All training together RAw", "All training together RAW"),
        ("All training together RAW", "All training together RAw"),
        ("bLINDED", "Blinded"),
        ("Blinded", "bLINDED"),
        ("Blinnded", "Blinded"),
        ("Blinded", "Blinnded"),
        ("Blinded folder", "Blinded"),
        ("Blinded", "Blinded folder"),
        ("Blinded Topomaps", "Blinded"),
        ("Blinded", "Blinded Topomaps"),
        ("TRAINING_TESTING_SPLIT", "TRAINING_TESTING_SPLIT_seed_99"),
        ("TRAINING_TESTING_SPLIT_seed_99", "TRAINING_TESTING_SPLIT"),
    ]

    for old, new in replacements:
        if old in csv_path:
            candidates.append(csv_path.replace(old, new))

    return unique_keep_order(candidates)


def patient_id_from_dataset_name(dataset_name):
    match = re.search(r"patient\s+(\d+)", str(dataset_name), flags=re.IGNORECASE)
    if match is None:
        return None
    return match.group(1).lstrip("0") or "0"


def score_csv_candidate(csv_path, patient_id):
    basename = os.path.basename(csv_path).lower()
    score = 0

    if "blind" in basename:
        score += 20
    if "raw" in basename or "a1" in basename:
        score += 10
    if "label" in basename:
        score += 5

    if patient_id is not None:
        patient_tokens = {
            patient_id,
            patient_id.zfill(2),
            f"patient_{patient_id}",
            f"patient_{patient_id.zfill(2)}",
            f"patient{patient_id}",
            f"patient{patient_id.zfill(2)}",
        }
        if any(token.lower() in basename for token in patient_tokens):
            score += 50

    return score


def resolve_csv_path(csv_path, dataset_name):
    csv_path = clean_path(csv_path)

    for candidate in csv_path_spelling_variants(csv_path):
        if os.path.isfile(candidate):
            if candidate != csv_path:
                print("\nCSV path repaired:")
                print("Original:", csv_path)
                print("Using   :", candidate)
            return candidate

    patient_id = patient_id_from_dataset_name(dataset_name)

    possible_parent_dirs = []
    for candidate in csv_path_spelling_variants(csv_path):
        possible_parent_dirs.append(os.path.dirname(candidate))

    possible_csvs = []
    for parent_dir in unique_keep_order(possible_parent_dirs):
        if not os.path.isdir(parent_dir):
            continue

        for filename in os.listdir(parent_dir):
            if filename.lower().endswith(".csv"):
                possible_csvs.append(os.path.join(parent_dir, filename))

    possible_csvs = unique_keep_order(possible_csvs)

    if len(possible_csvs) == 1:
        print("\nCSV filename repaired from the containing folder:")
        print("Original:", csv_path)
        print("Using   :", possible_csvs[0])
        return possible_csvs[0]

    if len(possible_csvs) > 1:
        scored = sorted(
            [
                (score_csv_candidate(candidate, patient_id), candidate)
                for candidate in possible_csvs
            ],
            reverse=True,
        )

        best_score, best_candidate = scored[0]
        tied_best = [
            candidate
            for score, candidate in scored
            if score == best_score
        ]

        if best_score > 0 and len(tied_best) == 1:
            print("\nCSV filename repaired from the containing folder:")
            print("Original:", csv_path)
            print("Using   :", best_candidate)
            return best_candidate

        print("\nCSV does not exist, and multiple CSV files were found in the folder.")
        print("Please choose the correct one from these candidates:")
        for _, candidate in scored:
            print(candidate)

    raise FileNotFoundError(f"\nCSV does not exist:\n{csv_path}")


def unique_keep_order(values):
    seen = set()
    ordered = []

    for value in values:
        if value is None:
            continue

        value = str(value).strip()
        if not value:
            continue

        if value not in seen:
            seen.add(value)
            ordered.append(value)

    return ordered


def filename_candidates_from_row(row, image_column):
    candidates = []

    for column in row.index:
        lower = str(column).lower()
        if "filename" in lower or lower in {"file", "image_name", "image"}:
            value = clean_path(row[column])
            if value:
                candidates.append(os.path.basename(value))

    original_path = clean_path(row[image_column])
    if original_path:
        candidates.append(os.path.basename(original_path))

    return unique_keep_order(candidates)


def class_folder_candidates_from_row(row, normalized_label):
    candidates = []

    for column in row.index:
        lower = str(column).lower()
        if "class" in lower or "category" in lower:
            value = str(row[column]).strip()
            if value and value.lower() != "nan":
                candidates.append(value)
                candidates.append(value.replace("-", "_").replace(" ", "_"))
                candidates.append(value.replace("_", "-").replace(" ", "-"))
                candidates.append(value.replace("_", " ").replace("-", " "))

    if int(normalized_label) == 1:
        candidates.extend(["Seizure", "seizure", "SEIZURE"])
    else:
        candidates.extend(
            [
                "Non_Seizure",
                "Non-Seizure",
                "Non Seizure",
                "Non-seizure",
                "non_seizure",
                "non-seizure",
                "non seizure",
            ]
        )

    return unique_keep_order(candidates)


def resolve_image_path_from_metadata(row, csv_path, image_column, normalized_label):
    original_path = clean_path(row[image_column])
    if original_path and os.path.isfile(original_path):
        return original_path, False

    csv_dir = os.path.dirname(os.path.abspath(csv_path))
    candidate_roots = unique_keep_order(
        [
            csv_dir,
            os.path.dirname(csv_dir),
            os.path.join(csv_dir, "Blinded"),
            os.path.join(csv_dir, "bLINDED"),
            os.path.join(csv_dir, "Testing"),
            os.path.join(csv_dir, "Training"),
        ]
    )

    filenames = filename_candidates_from_row(row, image_column)
    class_folders = class_folder_candidates_from_row(row, normalized_label)

    candidate_paths = []

    for root in candidate_roots:
        for filename in filenames:
            candidate_paths.append(os.path.join(root, filename))
            for class_folder in class_folders:
                candidate_paths.append(os.path.join(root, class_folder, filename))

    for candidate_path in unique_keep_order(candidate_paths):
        candidate_path = os.path.normpath(candidate_path)
        if os.path.isfile(candidate_path):
            return candidate_path, True

    return original_path, False


def detect_image_path_column(df, dataset_type):
    if dataset_type == "test":
        candidates = [
            "blinded_image_path",
            "blinded_topomap_path",
            "blinded_path",
            "blinded_image_filename",
            "blinded_topomap_filename",
            "fine_tuning_image_path",
            "image_path",
            "image_filename",
            "image_file",
            "image_name",
            "topomap_filename",
            "topomap_file",
            "topomap_name",
            "selected_image_path",
            "original_image_path",
            "file_path",
            "filepath",
            "filename",
            "file_name",
            "path",
        ]
    else:
        candidates = [
            "fine_tuning_image_path",
            "image_path",
            "image_filename",
            "image_file",
            "image_name",
            "topomap_filename",
            "topomap_file",
            "topomap_name",
            "selected_image_path",
            "blinded_image_path",
            "blinded_topomap_path",
            "blinded_path",
            "blinded_image_filename",
            "blinded_topomap_filename",
            "original_image_path",
            "file_path",
            "filepath",
            "filename",
            "file_name",
            "path",
        ]

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:
        lower = str(column).lower()
        if "path" in lower and ("image" in lower or "topomap" in lower or "blinded" in lower):
            return column
        if "filename" in lower and ("image" in lower or "topomap" in lower or "blind" in lower):
            return column

    raise ValueError(f"Could not detect image path column. Columns: {list(df.columns)}")


def detect_label_column(df):
    candidates = [
        "standard_label",
        "true_label",
        "label",
        "Label",
        "target",
        "true_class",
        "class_label",
        "class_name",
        "fine_tuning_class",
        "class_folder",
        "class",
        "category",
    ]

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:
        lower = str(column).lower()
        if "label" in lower or lower == "class":
            return column

    return None


def normalize_ground_truth_label(value):
    if pd.isna(value):
        return None

    if isinstance(value, (int, np.integer)):
        if int(value) == 1:
            return 1
        if int(value) == 0:
            return 0

    if isinstance(value, (float, np.floating)):
        if float(value) == 1.0:
            return 1
        if float(value) == 0.0:
            return 0

    text = str(value).strip().lower()
    text = text.replace("_", " ").replace("–", "-")
    text = re.sub(r"\s+", " ", text)

    if text in {"0", "0.0", "non seizure", "non-seizure", "nonseizure", "normal", "interictal", "ns"}:
        return 0
    if "non" in text and "seizure" in text:
        return 0
    if text in {"1", "1.0", "seizure", "ictal", "sz"}:
        return 1
    if "seizure" in text:
        return 1

    return None


def label_inference_columns(df):
    priority_columns = []

    for column in df.columns:
        lower = str(column).lower()

        if any(token in lower for token in ["label", "target", "category"]):
            priority_columns.append(column)
            continue

        if "class" in lower and "position" not in lower:
            priority_columns.append(column)
            continue

    for column in df.columns:
        lower = str(column).lower()

        if lower in {"filename", "file_name", "image_filename", "blinded_filename"}:
            priority_columns.append(column)
            continue

        if (
            ("filename" in lower or "path" in lower)
            and
            ("image" in lower or "topomap" in lower or "blind" in lower or "original" in lower)
        ):
            priority_columns.append(column)

    return unique_keep_order(priority_columns)


def infer_ground_truth_label_from_row(row, columns):
    for column in columns:
        label = normalize_ground_truth_label(row[column])
        if label is not None:
            return label

    return None


def load_and_standardize_metadata(csv_path, dataset_name, dataset_type):
    print("\n" + "=" * 100)
    print(f"LOADING {dataset_name.upper()}")
    print("=" * 100)

    csv_path = resolve_csv_path(csv_path, dataset_name)

    df = pd.read_csv(csv_path)
    print(f"\nRows: {len(df)}")
    print("\nColumns:")
    for column in df.columns:
        print("  ", column)

    image_column = detect_image_path_column(df, dataset_type)
    label_column = detect_label_column(df)
    print(f"\nImage path column: {image_column}")

    df["_image_path_original"] = df[image_column].apply(clean_path)

    if label_column is None:
        inference_columns = label_inference_columns(df)

        print("Label column     : inferred from metadata")
        print("Label inference columns:")
        for column in inference_columns:
            print("  ", column)

        if not inference_columns:
            raise ValueError(
                "Could not detect or infer a label column. "
                f"Columns: {list(df.columns)}"
            )

        df["_label"] = df.apply(
            lambda row: infer_ground_truth_label_from_row(row, inference_columns),
            axis=1
        )
    else:
        print(f"Label column     : {label_column}")
        df["_label"] = df[label_column].apply(normalize_ground_truth_label)

    invalid_labels = df[df["_label"].isna()]
    if len(invalid_labels) > 0:
        if label_column is None:
            preview_columns = label_inference_columns(df)
            print(invalid_labels[preview_columns].head(20))
        else:
            print(invalid_labels[[label_column]].drop_duplicates())
        raise ValueError("Some labels could not be interpreted.")

    df["_label"] = df["_label"].astype(int)

    resolved_paths = []
    repaired_count = 0

    for _, row in df.iterrows():
        resolved_path, repaired = resolve_image_path_from_metadata(
            row=row,
            csv_path=csv_path,
            image_column=image_column,
            normalized_label=int(row["_label"]),
        )
        resolved_paths.append(resolved_path)
        if repaired:
            repaired_count += 1

    df["_image_path"] = resolved_paths

    if repaired_count > 0:
        print(
            f"\nRepaired {repaired_count} image paths from the CSV folder "
            "using filename/class metadata."
        )

    missing_images = []
    for image_path in df["_image_path"]:
        if not os.path.isfile(image_path):
            missing_images.append(image_path)

    if missing_images:
        print(f"\nMissing images: {len(missing_images)}")
        print("\nThe CSV was found, but these image paths still do not exist.")
        print("Most likely the image_path column points to an old folder.")
        print("CSV folder used for automatic repair:")
        print(os.path.dirname(os.path.abspath(csv_path)))
        print("\nFirst missing paths:")
        for image_path in missing_images[:20]:
            print(image_path)
        raise FileNotFoundError("One or more images are missing.")

    duplicate_count = int(df["_image_path"].duplicated().sum())
    if duplicate_count > 0:
        raise ValueError(f"{duplicate_count} duplicate image paths detected in {dataset_name}.")

    seizure_count = int((df["_label"] == 1).sum())
    non_seizure_count = int((df["_label"] == 0).sum())

    print("\nClass counts:")
    print(f"Seizure     : {seizure_count}")
    print(f"Non-seizure : {non_seizure_count}")
    print(f"Total       : {len(df)}")
    print(f"\nAll {len(df)} image paths exist.")

    return df, image_column, label_column


full_training_df, train_image_column, train_label_column = load_and_standardize_metadata(
    TRAINING_METADATA_CSV,
    "pooled A2D2 training pool",
    "train",
)

if len(full_training_df) != EXPECTED_FULL_TRAIN_TOTAL:
    raise AssertionError(
        f"The pooled training CSV should contain {EXPECTED_FULL_TRAIN_TOTAL} rows, "
        f"but found {len(full_training_df)}."
    )

testing_dfs = {}
testing_column_info = {}

for patient_id, csv_path in TEST_METADATA_CSVS.items():
    test_df, image_column, label_column = load_and_standardize_metadata(
        csv_path,
        f"Patient {patient_id} A2D2 blinded test",
        "test",
    )
    testing_dfs[patient_id] = test_df
    testing_column_info[patient_id] = {
        "csv_path": csv_path,
        "image_column": image_column,
        "label_column": label_column,
    }


# %%
# ==================================================================================================
# 6. TRAIN / VALIDATION SPLIT + LEAKAGE CHECKS
# ==================================================================================================

seizure_pool = full_training_df[full_training_df["_label"] == 1].copy()
non_seizure_pool = full_training_df[full_training_df["_label"] == 0].copy()

seizure_pool = seizure_pool.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
non_seizure_pool = non_seizure_pool.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

train_seizure_count = int(round(len(seizure_pool) * TRAIN_FRACTION))
train_non_seizure_count = int(round(len(non_seizure_pool) * TRAIN_FRACTION))
validation_seizure_count = len(seizure_pool) - train_seizure_count
validation_non_seizure_count = len(non_seizure_pool) - train_non_seizure_count

actual_train_df = pd.concat(
    [
        seizure_pool.iloc[:train_seizure_count].copy(),
        non_seizure_pool.iloc[:train_non_seizure_count].copy(),
    ],
    ignore_index=True,
)

validation_df = pd.concat(
    [
        seizure_pool.iloc[train_seizure_count:].copy(),
        non_seizure_pool.iloc[train_non_seizure_count:].copy(),
    ],
    ignore_index=True,
)

actual_train_df = actual_train_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
validation_df = validation_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

if len(actual_train_df) != EXPECTED_ACTUAL_TRAIN_TOTAL:
    raise AssertionError(f"Expected {EXPECTED_ACTUAL_TRAIN_TOTAL} training images, found {len(actual_train_df)}.")
if len(validation_df) != EXPECTED_VALIDATION_TOTAL:
    raise AssertionError(f"Expected {EXPECTED_VALIDATION_TOTAL} validation images, found {len(validation_df)}.")

train_validation_overlap = set(actual_train_df["_image_path"]).intersection(set(validation_df["_image_path"]))
if train_validation_overlap:
    raise RuntimeError(f"Train/validation leakage detected: {len(train_validation_overlap)} identical image paths.")

for patient_id, test_df in testing_dfs.items():
    train_test_overlap = set(full_training_df["_image_path"]).intersection(set(test_df["_image_path"]))
    if train_test_overlap:
        raise RuntimeError(
            f"Training/test leakage detected for Patient {patient_id}: "
            f"{len(train_test_overlap)} identical image paths."
        )

print("\n" + "=" * 100)
print("POOLED A2D2 TRAIN / VALIDATION SPLIT")
print("=" * 100)
print(
    f"\nTRAIN: {int((actual_train_df['_label'] == 1).sum())} seizure + "
    f"{int((actual_train_df['_label'] == 0).sum())} non-seizure = {len(actual_train_df)}"
)
print(
    f"VALIDATION: {int((validation_df['_label'] == 1).sum())} seizure + "
    f"{int((validation_df['_label'] == 0).sum())} non-seizure = {len(validation_df)}"
)
print("\nTraining/validation/test current image-path overlap: 0")

actual_train_df.to_csv(TRAIN_SPLIT_CSV, index=False)
validation_df.to_csv(VALIDATION_SPLIT_CSV, index=False)


# %%
# ==================================================================================================
# 7. DATASET, PROCESSOR, BASE MODEL
# ==================================================================================================

class EEGTopomapDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        return {
            "index": int(index),
            "image_path": str(row["_image_path"]),
            "label": int(row["_label"]),
        }


print("\n" + "=" * 100)
print("LOADING MEDGEMMA PROCESSOR")
print("=" * 100)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=True)
if hasattr(processor, "tokenizer"):
    processor.tokenizer.padding_side = "right"

print("\nProcessor loaded successfully.")


def load_medgemma_base():
    try:
        loaded_model = AutoModelForMultimodalLM.from_pretrained(
            MODEL_ID,
            token=True,
            dtype=MODEL_DTYPE,
            low_cpu_mem_usage=True,
            attn_implementation="sdpa",
        )
    except TypeError:
        loaded_model = AutoModelForMultimodalLM.from_pretrained(
            MODEL_ID,
            token=True,
            torch_dtype=MODEL_DTYPE,
            low_cpu_mem_usage=True,
            attn_implementation="sdpa",
        )
    return loaded_model.to(DEVICE)


print("\n" + "=" * 100)
print("LOADING MEDGEMMA-4B")
print("=" * 100)

gc.collect()
torch.cuda.empty_cache()

base_model = load_medgemma_base()
print("\nSUCCESS: MedGemma-4B loaded.")
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

if hasattr(base_model.config, "use_cache"):
    base_model.config.use_cache = False

for parameter in base_model.parameters():
    parameter.requires_grad = False

if hasattr(base_model, "gradient_checkpointing_disable"):
    try:
        base_model.gradient_checkpointing_disable()
    except Exception:
        pass

if hasattr(base_model.config, "gradient_checkpointing"):
    try:
        base_model.config.gradient_checkpointing = False
    except Exception:
        pass

if bool(getattr(base_model, "is_gradient_checkpointing", False)):
    raise RuntimeError("Gradient checkpointing is still active; LoReFT hook run requires it OFF.")

print("Gradient checkpointing disabled.")
print("All pretrained MedGemma parameters frozen.")


# %%
# ==================================================================================================
# 8. LoReFT IMPLEMENTATION
# ==================================================================================================

def find_language_layers(multimodal_model):
    candidates = []

    if hasattr(multimodal_model, "model"):
        outer = multimodal_model.model
        if hasattr(outer, "language_model"):
            candidates.append(("model.language_model", outer.language_model))
        if hasattr(outer, "text_model"):
            candidates.append(("model.text_model", outer.text_model))

    if hasattr(multimodal_model, "language_model"):
        candidates.append(("language_model", multimodal_model.language_model))

    if hasattr(multimodal_model, "text_model"):
        candidates.append(("text_model", multimodal_model.text_model))

    for base_path, language_model in candidates:
        if hasattr(language_model, "layers") and isinstance(language_model.layers, nn.ModuleList):
            return language_model, language_model.layers, f"{base_path}.layers"

        if hasattr(language_model, "model"):
            nested = language_model.model
            if hasattr(nested, "layers") and isinstance(nested.layers, nn.ModuleList):
                return language_model, nested.layers, f"{base_path}.model.layers"

    for module_name, module in multimodal_model.named_modules():
        if (
            isinstance(module, nn.ModuleList)
            and module_name.endswith("layers")
            and "language_model" in module_name.lower()
            and "vision" not in module_name.lower()
        ):
            return None, module, module_name

    raise RuntimeError("Could not locate MedGemma's language transformer stack.")


class LowRankRotateLayer(nn.Module):
    def __init__(self, input_dim, rank):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(input_dim, rank), requires_grad=True)
        nn.init.orthogonal_(self.weight)

    def forward(self, x):
        return torch.matmul(x.to(self.weight.dtype), self.weight)


class LoReFTIntervention(nn.Module):
    def __init__(self, embed_dim, rank, train_dtype=torch.float32, dropout=0.0):
        super().__init__()

        rotate_layer = LowRankRotateLayer(embed_dim, rank).to(dtype=train_dtype)
        self.rotate_layer = nn.utils.parametrizations.orthogonal(rotate_layer)
        self.learned_source = nn.Linear(embed_dim, rank, bias=True).to(dtype=train_dtype)
        self.dropout = nn.Dropout(dropout)

        with torch.no_grad():
            current_rotation = self.rotate_layer.weight.detach().to(
                dtype=self.learned_source.weight.dtype,
                device=self.learned_source.weight.device,
            )
            self.learned_source.weight.copy_(current_rotation.T)
            self.learned_source.bias.zero_()

        self._capture_precision_diagnostic = False
        self._last_precision_diagnostic = None

    def arm_precision_diagnostic(self):
        self._capture_precision_diagnostic = True
        self._last_precision_diagnostic = None

    def get_precision_diagnostic(self):
        return self._last_precision_diagnostic

    def forward(self, base):
        original_dtype = base.dtype
        autocast_device_type = "cuda" if base.is_cuda else "cpu"

        with torch.autocast(device_type=autocast_device_type, enabled=False):
            base_fp32 = base.float()

            if self.rotate_layer.weight.dtype != torch.float32:
                raise RuntimeError(f"LoReFT rotate_layer weight is not FP32: {self.rotate_layer.weight.dtype}")
            if self.learned_source.weight.dtype != torch.float32:
                raise RuntimeError(f"LoReFT learned_source weight is not FP32: {self.learned_source.weight.dtype}")

            rotated_base_fp32 = self.rotate_layer(base_fp32)
            learned_source_fp32 = self.learned_source(base_fp32)
            delta_fp32 = torch.matmul(
                learned_source_fp32 - rotated_base_fp32,
                self.rotate_layer.weight.T,
            )
            updated_fp32 = base_fp32 + delta_fp32

        updated = updated_fp32.to(dtype=original_dtype)

        if self._capture_precision_diagnostic:
            with torch.no_grad():
                base_detached_fp32 = base.detach().float()
                raw_delta = delta_fp32.detach().float()
                updated_detached_fp32 = updated.detach().float()
                effective_delta = updated_detached_fp32 - base_detached_fp32

                raw_norm = float(raw_delta.norm().cpu())
                effective_norm = float(effective_delta.norm().cpu())
                cast_error_norm = float((raw_delta - effective_delta).norm().cpu())
                changed_fraction = float((updated.detach() != base.detach()).float().mean().cpu())
                retained_ratio = effective_norm / raw_norm if raw_norm > 0.0 else 0.0

                self._last_precision_diagnostic = {
                    "base_dtype": str(base.dtype),
                    "reft_dtype": str(delta_fp32.dtype),
                    "raw_fp32_delta_norm": raw_norm,
                    "effective_post_cast_delta_norm": effective_norm,
                    "cast_error_norm": cast_error_norm,
                    "retained_norm_ratio": retained_ratio,
                    "changed_element_fraction": changed_fraction,
                    "raw_fp32_delta_mean_abs": float(raw_delta.abs().mean().cpu()),
                    "effective_post_cast_delta_mean_abs": float(effective_delta.abs().mean().cpu()),
                }

            self._capture_precision_diagnostic = False

        return self.dropout(updated)


class LoReFTPositionController:
    def __init__(self, prefix_tokens, suffix_tokens):
        self.prefix_tokens = int(prefix_tokens)
        self.suffix_tokens = int(suffix_tokens)
        self.active = False
        self.prompt_length = None
        self.minimum_sequence_length = None

    def activate(self, prompt_length, minimum_sequence_length=None):
        self.prompt_length = int(prompt_length)
        self.minimum_sequence_length = int(minimum_sequence_length if minimum_sequence_length is not None else prompt_length)
        self.active = True

    def deactivate(self):
        self.active = False
        self.prompt_length = None
        self.minimum_sequence_length = None

    def get_positions(self, current_sequence_length):
        if not self.active:
            return []
        if current_sequence_length < self.minimum_sequence_length:
            return []

        prompt_length = min(self.prompt_length, int(current_sequence_length))
        prefix = list(range(min(self.prefix_tokens, prompt_length)))
        suffix = list(range(max(0, prompt_length - self.suffix_tokens), prompt_length))

        positions = []
        seen = set()
        for position in prefix + suffix:
            if position not in seen:
                seen.add(position)
                positions.append(position)

        return positions


def _extract_hidden(layer_output):
    if torch.is_tensor(layer_output):
        return layer_output
    if isinstance(layer_output, (tuple, list)) and len(layer_output) > 0 and torch.is_tensor(layer_output[0]):
        return layer_output[0]
    raise TypeError(f"Unsupported language-layer output type: {type(layer_output)}")


def _replace_hidden(layer_output, new_hidden):
    if torch.is_tensor(layer_output):
        return new_hidden
    if isinstance(layer_output, tuple):
        return (new_hidden,) + tuple(layer_output[1:])
    if isinstance(layer_output, list):
        return [new_hidden] + list(layer_output[1:])
    raise TypeError(f"Unsupported language-layer output type: {type(layer_output)}")


def make_loreft_hook(intervention, controller):
    def hook(module, inputs, output):
        hidden = _extract_hidden(output)

        if hidden.ndim != 3:
            raise RuntimeError(f"Expected [batch, sequence, hidden], got {tuple(hidden.shape)}")

        positions = controller.get_positions(hidden.shape[1])
        if not positions:
            return output

        position_tensor = torch.tensor(positions, dtype=torch.long, device=hidden.device)
        selected = hidden.index_select(1, position_tensor)
        modified = intervention(selected)

        new_hidden = hidden.clone()
        new_hidden[:, position_tensor, :] = modified
        return _replace_hidden(output, new_hidden)

    return hook


class MedGemmaLoReFTWrapper(nn.Module):
    def __init__(self, base, controller, interventions, checkpoint_info):
        super().__init__()
        self.base_model = base
        self.controller = controller
        self.checkpoint_info = dict(checkpoint_info)
        object.__setattr__(self, "_interventions_ref", interventions)

    @property
    def config(self):
        return self.base_model.config

    def forward(self, *args, **kwargs):
        labels = kwargs.get("labels", None)

        if labels is not None:
            if labels.ndim != 2 or labels.shape[0] != 1:
                raise ValueError("This LoReFT pipeline expects physical batch size = 1.")

            supervised_positions = torch.nonzero(labels[0] != -100, as_tuple=False).flatten()
            if supervised_positions.numel() == 0:
                raise RuntimeError("No assistant target token was found in the loss mask.")

            prompt_length = int(supervised_positions[0].item())
        else:
            input_ids = kwargs.get("input_ids", None)
            if input_ids is None:
                raise RuntimeError("LoReFT forward needs labels or input_ids to determine prompt positions.")
            prompt_length = int(input_ids.shape[1])

        self.controller.activate(prompt_length=prompt_length, minimum_sequence_length=prompt_length)
        return self.base_model(*args, **kwargs)

    @torch.no_grad()
    def generate(self, *args, **kwargs):
        input_ids = kwargs.get("input_ids", None)
        if input_ids is None:
            raise RuntimeError("generate() requires input_ids for LoReFT position control.")

        prompt_length = int(input_ids.shape[1])
        self.controller.activate(prompt_length=prompt_length, minimum_sequence_length=prompt_length)

        try:
            return self.base_model.generate(*args, **kwargs)
        finally:
            self.controller.deactivate()

    def save_pretrained(self, save_directory, safe_serialization=True, **kwargs):
        os.makedirs(save_directory, exist_ok=True)

        multi_layer_state = {
            layer_key: intervention.state_dict()
            for layer_key, intervention in self._interventions_ref.items()
        }

        torch.save(multi_layer_state, os.path.join(save_directory, "loreft_intervention.pt"))

        with open(os.path.join(save_directory, "loreft_config.json"), "w", encoding="utf-8") as file:
            json.dump(self.checkpoint_info, file, indent=4)


def resolve_loreft_model_info(multimodal_model, target_layers_human):
    language_model_object, language_layers, language_layers_path = find_language_layers(multimodal_model)
    num_language_layers = len(language_layers)

    target_layers_human = [int(x) for x in target_layers_human]
    if len(target_layers_human) != len(set(target_layers_human)):
        raise ValueError("Duplicate LoReFT target layers were requested.")

    for human_layer in target_layers_human:
        if human_layer < 1 or human_layer > num_language_layers:
            raise ValueError(f"Requested LoReFT layer {human_layer}, but the language model has {num_language_layers} layers.")

    target_layer_indices = [human_layer - 1 for human_layer in target_layers_human]

    if hasattr(multimodal_model.config, "text_config") and hasattr(multimodal_model.config.text_config, "hidden_size"):
        hidden_size = int(multimodal_model.config.text_config.hidden_size)
    elif language_model_object is not None and hasattr(language_model_object, "config"):
        hidden_size = int(language_model_object.config.hidden_size)
    else:
        raise RuntimeError("Could not determine the language hidden size.")

    return {
        "language_model_object": language_model_object,
        "language_layers": language_layers,
        "language_layers_path": language_layers_path,
        "num_language_layers": num_language_layers,
        "hidden_size": hidden_size,
        "target_layers_human": target_layers_human,
        "target_layer_indices": target_layer_indices,
    }


def attach_loreft(
    base,
    target_layers_human,
    rank,
    prefix_tokens,
    suffix_tokens,
    dropout,
    trainable,
    saved_state=None,
    saved_config=None,
):
    info = resolve_loreft_model_info(base, target_layers_human)
    language_layers = info["language_layers"]
    target_layer_indices = info["target_layer_indices"]

    controller = LoReFTPositionController(prefix_tokens=prefix_tokens, suffix_tokens=suffix_tokens)
    target_language_layers = []
    interventions = {}
    hook_handles = []

    for human_layer, python_index in zip(info["target_layers_human"], target_layer_indices):
        layer_key = str(human_layer)
        target_layer = language_layers[python_index]

        intervention = LoReFTIntervention(
            embed_dim=info["hidden_size"],
            rank=rank,
            train_dtype=torch.float32,
            dropout=dropout,
        ).to(DEVICE)

        target_layer.add_module(LOREFT_MODULE_NAME, intervention)

        if saved_state is not None:
            if layer_key not in saved_state:
                raise RuntimeError(f"Missing saved LoReFT state for human layer {human_layer}.")
            intervention.load_state_dict(saved_state[layer_key], strict=True)

        for parameter in intervention.parameters():
            parameter.requires_grad = bool(trainable)

        hook_handle = target_layer.register_forward_hook(make_loreft_hook(intervention, controller))

        target_language_layers.append(target_layer)
        interventions[layer_key] = intervention
        hook_handles.append(hook_handle)

    checkpoint_info = {
        "method": "LoReFT",
        "variant": "multi_layer_independent_interventions",
        "formula": "h + R^T(Wh + b - Rh)",
        "rank": rank,
        "num_language_layers": info["num_language_layers"],
        "target_layers_human": info["target_layers_human"],
        "target_layer_python_indices": target_layer_indices,
        "num_interventions": len(target_layer_indices),
        "hidden_size": info["hidden_size"],
        "prefix_prompt_tokens": prefix_tokens,
        "suffix_prompt_tokens": suffix_tokens,
        "dropout": dropout,
        "component": LOREFT_COMPONENT,
        "language_layers_path": info["language_layers_path"],
        "precision_fix": "autocast disabled inside LoReFT; true FP32 projection + FP32 residual addition; one final cast to model dtype",
        "initialization": "identity_no_op_W_equals_R_transpose_bias_zero",
        "bf16_diagnostic": True,
        "rotation_weight_decay": 0.0,
        "source_weight_decay": WEIGHT_DECAY,
        "source_bias_weight_decay": 0.0,
    }

    if saved_config is not None:
        checkpoint_info.update(saved_config)

    wrapper = MedGemmaLoReFTWrapper(
        base=base,
        controller=controller,
        interventions=interventions,
        checkpoint_info=checkpoint_info,
    ).to(DEVICE)

    return wrapper, info, controller, interventions, hook_handles, target_language_layers, checkpoint_info


model, loreft_info, loreft_controller, loreft_interventions, loreft_hook_handles, target_language_layers, checkpoint_info = attach_loreft(
    base=base_model,
    target_layers_human=LOREFT_TARGET_LAYERS_HUMAN,
    rank=LOREFT_RANK,
    prefix_tokens=LOREFT_PREFIX_TOKENS,
    suffix_tokens=LOREFT_SUFFIX_TOKENS,
    dropout=LOREFT_DROPOUT,
    trainable=True,
)

NUM_LANGUAGE_LAYERS = loreft_info["num_language_layers"]
LOREFT_HIDDEN_SIZE = loreft_info["hidden_size"]
LANGUAGE_LAYERS_PATH = loreft_info["language_layers_path"]
LOREFT_TARGET_LAYER_INDICES = loreft_info["target_layer_indices"]

print("\n" + "=" * 100)
print("MEDGEMMA LANGUAGE STACK")
print("=" * 100)
print(f"Language layer path        : {LANGUAGE_LAYERS_PATH}")
print(f"Number of language layers  : {NUM_LANGUAGE_LAYERS}")
print(f"Language hidden size       : {LOREFT_HIDDEN_SIZE}")
print(f"Intervention human layers  : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Intervention Python indices: {LOREFT_TARGET_LAYER_INDICES}")
print(f"Number interventions       : {len(LOREFT_TARGET_LAYER_INDICES)}")


# %%
# ==================================================================================================
# 9. TRAINABLE PARAMETER CHECKS
# ==================================================================================================

trainable_named_parameters = [
    (name, parameter)
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

unexpected_trainable = [
    name
    for name, parameter in trainable_named_parameters
    if LOREFT_MODULE_NAME not in name
]

if unexpected_trainable:
    raise RuntimeError("Unexpected pretrained parameters are trainable:\n" + "\n".join(unexpected_trainable[:50]))

if not trainable_named_parameters:
    raise RuntimeError("No LoReFT parameters are trainable.")

non_fp32_trainable = [
    f"{name}: {parameter.dtype}"
    for name, parameter in trainable_named_parameters
    if parameter.dtype != torch.float32
]

if non_fp32_trainable:
    raise RuntimeError(
        "Every trainable LoReFT parameter must remain FP32, but these do not:\n"
        + "\n".join(non_fp32_trainable)
    )

trainable_parameter_count = sum(parameter.numel() for name, parameter in trainable_named_parameters)
total_parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_percentage = 100.0 * trainable_parameter_count / total_parameter_count

expected_per_intervention = 2 * LOREFT_HIDDEN_SIZE * LOREFT_RANK + LOREFT_RANK
expected_total_loreft_parameters = expected_per_intervention * len(LOREFT_TARGET_LAYER_INDICES)

if trainable_parameter_count != expected_total_loreft_parameters:
    raise RuntimeError(
        "Unexpected multi-layer LoReFT trainable-parameter count.\n"
        f"Expected: {expected_total_loreft_parameters:,}\n"
        f"Found   : {trainable_parameter_count:,}"
    )

print("\n" + "=" * 100)
print("MULTI-LAYER LoReFT TRAINABLE PARAMETER REPORT")
print("=" * 100)
for name, parameter in trainable_named_parameters:
    print(f"  {name}: {parameter.numel():,}")

print(f"\nIntervention layers  : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Parameters / layer   : {expected_per_intervention:,}")
print(f"Trainable parameters : {trainable_parameter_count:,}")
print(f"Total parameters     : {total_parameter_count:,}")
print(f"Trainable percentage : {trainable_percentage:.6f}%")
print("LoReFT parameter dtype: torch.float32")
print("LoReFT compute dtype  : torch.float32")
print("Initialization        : identity/no-op (W = R^T, b = 0)")

with open(TARGET_MODULES_TXT, "w", encoding="utf-8") as file:
    for key, value in checkpoint_info.items():
        file.write(f"{key}={value}\n")


# %%
# ==================================================================================================
# 10. PROMPT, COLLATOR, LOADERS
# ==================================================================================================

def label_to_target_text(label):
    if int(label) == 1:
        return "Seizure"
    return "Non-seizure"


def build_prompt_messages(image):
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_PROMPT}],
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": USER_PROMPT},
                {"type": "image", "image": image},
            ],
        },
    ]


def build_training_messages(image, target_text):
    messages = build_prompt_messages(image)
    messages.append(
        {
            "role": "assistant",
            "content": [{"type": "text", "text": target_text}],
        }
    )
    return messages


class MedGemmaTrainingCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        if len(examples) != 1:
            raise ValueError("This pipeline expects physical batch size = 1.")

        example = examples[0]
        image_path = example["image_path"]
        target_text = label_to_target_text(example["label"])

        with Image.open(image_path) as opened_image:
            image = opened_image.convert("RGB")

            prompt_messages = build_prompt_messages(image)
            prompt_inputs = self.processor.apply_chat_template(
                prompt_messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )

            full_messages = build_training_messages(image, target_text)
            full_inputs = self.processor.apply_chat_template(
                full_messages,
                add_generation_prompt=False,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )

        prompt_ids = prompt_inputs["input_ids"]
        full_ids = full_inputs["input_ids"]
        prompt_length = int(prompt_ids.shape[1])

        if full_ids.shape[1] <= prompt_length:
            raise RuntimeError("Full training conversation contains no target response.")

        prefix_matches = torch.equal(full_ids[:, :prompt_length], prompt_ids)
        if not prefix_matches:
            raise RuntimeError("MedGemma chat template prefix mismatch. Training stopped.")

        labels = full_ids.clone()
        labels[:, :prompt_length] = -100

        if int((labels != -100).sum()) == 0:
            raise RuntimeError("No assistant target tokens remain after masking.")

        full_inputs["labels"] = labels
        return full_inputs


train_dataset = EEGTopomapDataset(actual_train_df)
validation_dataset = EEGTopomapDataset(validation_df)
training_collator = MedGemmaTrainingCollator(processor)

train_generator = torch.Generator()
train_generator.manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=NUM_WORKERS,
    collate_fn=training_collator,
)

validation_loss_loader = DataLoader(
    validation_dataset,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=training_collator,
)


def move_batch_to_device(batch):
    moved = {}
    for key, value in batch.items():
        if not torch.is_tensor(value):
            moved[key] = value
        elif value.dtype.is_floating_point:
            moved[key] = value.to(device=DEVICE, dtype=MODEL_DTYPE)
        else:
            moved[key] = value.to(DEVICE)
    return moved


def autocast_context():
    return torch.autocast(device_type="cuda", dtype=MODEL_DTYPE)


# %%
# ==================================================================================================
# 11. DIAGNOSTICS, INFERENCE, METRICS
# ==================================================================================================

def arm_all_precision_diagnostics(interventions):
    for intervention in interventions.values():
        intervention.arm_precision_diagnostic()


def print_all_precision_diagnostics(interventions, title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    for layer_key in sorted(interventions.keys(), key=lambda x: int(x)):
        stats = interventions[layer_key].get_precision_diagnostic()
        if stats is None:
            print(f"Layer {layer_key}: no diagnostic sample was captured.")
            continue

        print(f"Layer {layer_key}:")
        print(f"  Base hidden dtype             : {stats['base_dtype']}")
        print(f"  LoReFT computation dtype      : {stats['reft_dtype']}")
        print(f"  Raw FP32 delta L2 norm        : {stats['raw_fp32_delta_norm']:.12e}")
        print(f"  Effective post-cast delta norm: {stats['effective_post_cast_delta_norm']:.12e}")
        print(f"  BF16 cast error L2 norm       : {stats['cast_error_norm']:.12e}")
        print(f"  Retained delta-norm ratio     : {stats['retained_norm_ratio']:.6f}")
        print(f"  Hidden elements changed       : {100.0 * stats['changed_element_fraction']:.4f}%")
        print(f"  Raw FP32 mean |delta|         : {stats['raw_fp32_delta_mean_abs']:.12e}")
        print(f"  Effective mean |delta|        : {stats['effective_post_cast_delta_mean_abs']:.12e}")

    print("=" * 100)


def normalize_prediction_text(output_text):
    text = str(output_text).strip().lower()
    text = text.replace("_", " ").replace("–", "-")
    text = re.sub(r"\s+", " ", text)

    if re.search(r"\bnon[\s-]*seizure\b", text):
        return 0
    if re.search(r"\bseizure\b", text):
        return 1
    return None


@torch.no_grad()
def predict_single_image(image_path):
    model.eval()

    with Image.open(image_path) as opened_image:
        image = opened_image.convert("RGB")
        messages = build_prompt_messages(image)
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

    input_length = int(inputs["input_ids"].shape[1])
    inputs = move_batch_to_device(inputs)

    with autocast_context():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    generated_tokens = generated[0, input_length:]
    decoded = processor.decode(generated_tokens, skip_special_tokens=True).strip()
    prediction = normalize_prediction_text(decoded)
    return decoded, prediction


def calculate_binary_metrics(true_labels, predictions):
    metric_predictions = []
    invalid_count = 0

    for truth, prediction in zip(true_labels, predictions):
        if prediction is None:
            invalid_count += 1
            metric_predictions.append(1 - int(truth))
        else:
            metric_predictions.append(int(prediction))

    y_true = np.asarray(true_labels, dtype=int)
    y_pred = np.asarray(metric_predictions, dtype=int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metrics = {
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Sensitivity": float(sensitivity),
        "Specificity": float(specificity),
        "Balanced_Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Invalid_Predictions": int(invalid_count),
    }

    return metrics, y_pred


@torch.no_grad()
def calculate_validation_loss():
    model.eval()
    losses = []

    progress = tqdm(validation_loss_loader, desc="Validation loss", leave=False)
    for batch in progress:
        batch = move_batch_to_device(batch)

        with autocast_context():
            outputs = model(**batch)

        loss_value = float(outputs.loss.detach().float().cpu())
        losses.append(loss_value)
        progress.set_postfix(loss=f"{loss_value:.4f}")

        del batch
        del outputs

    return float(np.mean(losses))


@torch.no_grad()
def evaluate_validation_generation(epoch):
    model.eval()
    true_labels = []
    predictions = []
    rows = []

    progress = tqdm(range(len(validation_df)), desc=f"Validation predictions epoch {epoch}")
    for index in progress:
        row = validation_df.iloc[index]
        image_path = str(row["_image_path"])

        raw_output, prediction = predict_single_image(image_path)
        true_label = int(row["_label"])

        true_labels.append(true_label)
        predictions.append(prediction)

        rows.append(
            {
                "index": index,
                "image_path": image_path,
                "true_label": true_label,
                "true_class": "Seizure" if true_label == 1 else "Non-seizure",
                "raw_model_output": raw_output,
                "prediction": prediction if prediction is not None else "INVALID",
                "predicted_class": (
                    "Seizure"
                    if prediction == 1
                    else "Non-seizure"
                    if prediction == 0
                    else "INVALID"
                ),
            }
        )

    metrics, metric_predictions = calculate_binary_metrics(true_labels, predictions)
    validation_prediction_df = pd.DataFrame(rows)
    validation_prediction_df["metric_prediction"] = metric_predictions
    validation_prediction_df["correct"] = validation_prediction_df["true_label"].to_numpy() == metric_predictions

    validation_output_csv = os.path.join(OUTPUT_ROOT, f"validation_predictions_epoch_{epoch}.csv")
    validation_prediction_df.to_csv(validation_output_csv, index=False)

    return metrics


# %%
# ==================================================================================================
# 12. PRE-FLIGHT TEST
# ==================================================================================================

print("\n" + "=" * 100)
print("LoReFT R4 PRE-FLIGHT TEST")
print("=" * 100)

preview_example = train_dataset[0]
print("\nImage:", preview_example["image_path"])
print("Target:", label_to_target_text(preview_example["label"]))

preview_batch_cpu = training_collator([preview_example])
target_ids = preview_batch_cpu["labels"][0]
target_ids = target_ids[target_ids != -100]

print("Assistant target tokens:", len(target_ids))

try:
    print("Decoded supervised target:", repr(processor.decode(target_ids, skip_special_tokens=False)))
except Exception:
    pass

preview_batch_gpu = move_batch_to_device(preview_batch_cpu)
model.eval()
arm_all_precision_diagnostics(loreft_interventions)

with torch.no_grad():
    with autocast_context():
        preview_output = model(**preview_batch_gpu)

preview_loss = float(preview_output.loss.detach().float().cpu())
print(f"Pre-flight loss: {preview_loss:.6f}")

print_all_precision_diagnostics(
    loreft_interventions,
    "PRE-FLIGHT TRUE-FP32 LoReFT INTERVENTION DIAGNOSTIC",
)

if not np.isfinite(preview_loss):
    raise RuntimeError("Pre-flight loss is not finite.")

print("\nSUCCESS: LoReFT R4 forward pass works.")

del preview_batch_cpu
del preview_batch_gpu
del preview_output
gc.collect()
torch.cuda.empty_cache()


# %%
# ==================================================================================================
# 13. OPTIMIZER, SCHEDULER, UPDATE DIAGNOSTICS
# ==================================================================================================

trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]

rotation_parameters = []
source_weight_parameters = []
source_bias_parameters = []

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    if "rotate_layer.parametrizations.weight.original" in name:
        rotation_parameters.append(parameter)
    elif name.endswith("learned_source.bias"):
        source_bias_parameters.append(parameter)
    elif "learned_source.weight" in name:
        source_weight_parameters.append(parameter)
    else:
        raise RuntimeError(f"Unexpected trainable LoReFT parameter in optimizer grouping: {name}")

if len(rotation_parameters) != NUM_LOREFT_INTERVENTIONS:
    raise RuntimeError(
        f"Expected one trainable orthogonal-rotation parameter per LoReFT layer. "
        f"Expected {NUM_LOREFT_INTERVENTIONS}, found {len(rotation_parameters)}."
    )

optimizer = torch.optim.AdamW(
    [
        {"params": rotation_parameters, "weight_decay": 0.0, "group_name": "loreft_rotation_no_decay"},
        {"params": source_weight_parameters, "weight_decay": WEIGHT_DECAY, "group_name": "loreft_source_weight"},
        {"params": source_bias_parameters, "weight_decay": 0.0, "group_name": "loreft_source_bias_no_decay"},
    ],
    lr=LEARNING_RATE,
)

print("\nLoReFT optimizer parameter groups:")
for group in optimizer.param_groups:
    print(
        f"  {group.get('group_name', 'unnamed')}: "
        f"weight_decay={group['weight_decay']}, "
        f"parameters={sum(p.numel() for p in group['params']):,}"
    )


def snapshot_trainable_parameters():
    return [parameter.detach().float().cpu().clone() for parameter in trainable_parameters]


def effective_rotation_norms():
    norms = {}
    for layer_key, intervention in loreft_interventions.items():
        with torch.no_grad():
            effective_rotation = intervention.rotate_layer.weight.detach().float()
            norms[layer_key] = float(effective_rotation.norm().cpu())
    return norms


def print_effective_rotation_norms(title):
    norms = effective_rotation_norms()
    expected = float(LOREFT_RANK ** 0.5)

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    for layer_key in sorted(norms.keys(), key=lambda x: int(x)):
        value = norms[layer_key]
        print(f"Layer {layer_key}: ||R||_F = {value:.12e} (expected ~= {expected:.6f})")

        if not np.isfinite(value) or value < 0.5 * expected:
            raise RuntimeError(
                f"Effective LoReFT rotation collapsed at layer {layer_key}. "
                f"Observed ||R||_F={value:.12e}, expected near {expected:.6f}."
            )

    print("=" * 100)


def trainable_gradient_l2_norm():
    total_sq = 0.0
    nonzero_tensors = 0
    tensors_with_grad = 0

    for parameter in trainable_parameters:
        if parameter.grad is None:
            continue

        tensors_with_grad += 1
        grad = parameter.grad.detach().float()
        total_sq += float(torch.sum(grad * grad).cpu())

        if bool(torch.any(grad != 0).item()):
            nonzero_tensors += 1

    return math.sqrt(total_sq), tensors_with_grad, nonzero_tensors


def parameter_change_from_snapshot(before_snapshot):
    total_sq = 0.0
    max_abs = 0.0

    for before, parameter in zip(before_snapshot, trainable_parameters):
        after = parameter.detach().float().cpu()
        delta = after - before
        total_sq += float(torch.sum(delta * delta))
        if delta.numel() > 0:
            max_abs = max(max_abs, float(delta.abs().max()))

    return math.sqrt(total_sq), max_abs


optimizer_steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
total_optimizer_steps = optimizer_steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_optimizer_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps,
)

if MODEL_DTYPE == torch.float16:
    try:
        grad_scaler = torch.amp.GradScaler("cuda")
    except Exception:
        grad_scaler = torch.cuda.amp.GradScaler()
else:
    grad_scaler = None

print("\n" + "=" * 100)
print("LoReFT R4 TRAINING SETTINGS")
print("=" * 100)
print(f"\nEpochs                    : {NUM_EPOCHS}")
print(f"Training images           : {len(train_dataset)}")
print(f"Validation images         : {len(validation_dataset)}")
print(f"Physical batch size       : {TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation     : {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size      : {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate             : {LEARNING_RATE}")
print(f"Warmup ratio              : {WARMUP_RATIO}")
print(f"Warmup optimizer steps    : {warmup_steps}")
print(f"LoReFT rank               : {LOREFT_RANK}")
print(f"Language layers           : {NUM_LANGUAGE_LAYERS}")
print(f"Intervention human layers : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Intervention Python idx   : {LOREFT_TARGET_LAYER_INDICES}")
print(f"Prefix prompt tokens      : {LOREFT_PREFIX_TOKENS}")
print(f"Suffix prompt tokens      : {LOREFT_SUFFIX_TOKENS}")
print(f"LoReFT dropout            : {LOREFT_DROPOUT}")
print("Base MedGemma weights     : FROZEN")
print("Vision tower              : FROZEN")
print("Gradient checkpointing    : OFF")
print("LoReFT parameter dtype    : float32")
print("LoReFT compute dtype      : float32")
print("Rotation weight decay     : 0.0")
print(f"Source weight decay       : {WEIGHT_DECAY}")
print(f"Run ID                    : {RUN_ID}")
print(f"Output folder             : {OUTPUT_ROOT}")

print_effective_rotation_norms("PRE-TRAIN EFFECTIVE LoReFT ROTATION-NORM CHECK")


# %%
# ==================================================================================================
# 14. SAVE EXPERIMENT CONFIGURATION
# ==================================================================================================

experiment_config = {
    "experiment": "Pooled_A2D2_R4",
    "patients_in_training_pool": ["08", "11", "18", "24"],
    "representation": "A2D2-coefficient fixed-scale EEG variance topomap",
    "model": MODEL_ID,
    "fine_tuning_method": "LoReFT",
    "loreft_formula": "h + R^T(Wh + b - Rh)",
    "quantization": "None",
    "random_seed": RANDOM_SEED,
    "training_metadata_csv": TRAINING_METADATA_CSV,
    "test_metadata_csvs": TEST_METADATA_CSVS,
    "system_prompt": SYSTEM_PROMPT,
    "user_prompt": USER_PROMPT,
    "training_pool_images": len(full_training_df),
    "actual_training_images": len(actual_train_df),
    "validation_images": len(validation_df),
    "loreft_rank": LOREFT_RANK,
    "num_language_layers": NUM_LANGUAGE_LAYERS,
    "loreft_target_layers_human": LOREFT_TARGET_LAYERS_HUMAN,
    "loreft_target_layer_python_indices": LOREFT_TARGET_LAYER_INDICES,
    "loreft_num_interventions": len(LOREFT_TARGET_LAYER_INDICES),
    "language_layers_path": LANGUAGE_LAYERS_PATH,
    "hidden_size": LOREFT_HIDDEN_SIZE,
    "prefix_prompt_tokens": LOREFT_PREFIX_TOKENS,
    "suffix_prompt_tokens": LOREFT_SUFFIX_TOKENS,
    "loreft_dropout": LOREFT_DROPOUT,
    "component": LOREFT_COMPONENT,
    "gradient_checkpointing": False,
    "loreft_train_dtype": "float32",
    "checkpoint_save_strategy": "new directory per best epoch; no rmtree",
    "epochs": NUM_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "best_model_primary_criterion": "Highest validation balanced accuracy",
    "best_model_tie_breaker": "Lower validation loss",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "physical_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "trainable_parameters": trainable_parameter_count,
    "total_parameters": total_parameter_count,
    "trainable_percentage": trainable_percentage,
    "precision": PRECISION_NAME,
    "gpu": GPU_NAME,
    "gpu_memory_gb": GPU_MEMORY_GB,
    "pytorch_version": torch.__version__,
    "transformers_version": transformers.__version__,
}

with open(EXPERIMENT_CONFIG_JSON, "w", encoding="utf-8") as file:
    json.dump(experiment_config, file, indent=4)


# %%
# ==================================================================================================
# 15. TRAINING LOOP
# ==================================================================================================

training_history = []
best_validation_balanced_accuracy = -float("inf")
best_validation_loss = float("inf")
best_epoch = None
epochs_without_improvement = 0
stopped_early = False
early_stop_epoch = None
first_nonzero_lr_update_checked = False
zero_lr_warmup_step_reported = False

optimizer.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats(DEVICE)
training_start_time = time.time()

print("\n" + "=" * 100)
print("STARTING POOLED A2D2 MEDGEMMA-4B LoReFT R4 TRAINING")
print("=" * 100)

for epoch in range(1, NUM_EPOCHS + 1):
    print("\n" + "=" * 100)
    print(f"EPOCH {epoch}/{NUM_EPOCHS}")
    print("=" * 100)

    model.train()
    epoch_losses = []
    epoch_start_time = time.time()
    epoch_parameter_snapshot = snapshot_trainable_parameters()

    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        enumerate(train_loader, start=1),
        total=len(train_loader),
        desc=f"LoReFT R4 training epoch {epoch}",
    )

    accumulation_count = 0

    for batch_number, batch in progress:
        try:
            batch = move_batch_to_device(batch)

            with autocast_context():
                outputs = model(**batch)
                raw_loss = outputs.loss
                scaled_loss = raw_loss / GRADIENT_ACCUMULATION_STEPS

            if not torch.isfinite(raw_loss):
                raise RuntimeError("Training loss became NaN/Inf.")

            if grad_scaler is not None:
                grad_scaler.scale(scaled_loss).backward()
            else:
                scaled_loss.backward()

            accumulation_count += 1
            current_loss = float(raw_loss.detach().float().cpu())
            epoch_losses.append(current_loss)

            should_update = accumulation_count == GRADIENT_ACCUMULATION_STEPS or batch_number == len(train_loader)

            if should_update:
                if grad_scaler is not None:
                    grad_scaler.unscale_(optimizer)

                gradient_l2, grad_tensor_count, nonzero_grad_tensors = trainable_gradient_l2_norm()
                optimizer_lr_before_step = float(optimizer.param_groups[0]["lr"])
                diagnostic_this_step = False

                if not first_nonzero_lr_update_checked and optimizer_lr_before_step > 0.0:
                    diagnostic_this_step = True
                    first_update_before = snapshot_trainable_parameters()

                    print("\n" + "=" * 100)
                    print("FIRST NON-ZERO-LR LoReFT UPDATE DIAGNOSTIC")
                    print("=" * 100)
                    print(f"Optimizer LR before step     : {optimizer_lr_before_step:.12e}")
                    print(f"Gradient L2 norm before clip : {gradient_l2:.12e}")
                    print(f"Gradient tensors present     : {grad_tensor_count}")
                    print(f"Gradient tensors non-zero    : {nonzero_grad_tensors}")

                    if grad_tensor_count == 0 or nonzero_grad_tensors == 0 or gradient_l2 <= 0.0:
                        raise RuntimeError("LoReFT received no non-zero gradient on the first non-zero-LR update.")

                elif (
                    not first_nonzero_lr_update_checked
                    and optimizer_lr_before_step == 0.0
                    and not zero_lr_warmup_step_reported
                ):
                    print("\n" + "=" * 100)
                    print("LoReFT WARMUP NOTE")
                    print("=" * 100)
                    print("Optimizer LR before step     : 0.000000000000e+00")
                    print(f"Gradient L2 norm before clip : {gradient_l2:.12e}")
                    print(f"Gradient tensors present     : {grad_tensor_count}")
                    print(f"Gradient tensors non-zero    : {nonzero_grad_tensors}")
                    print("The first optimizer step can use LR=0 because of linear warmup.")
                    print("=" * 100)
                    zero_lr_warmup_step_reported = True

                clipped_norm = torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)

                if grad_scaler is not None:
                    grad_scaler.step(optimizer)
                    grad_scaler.update()
                else:
                    optimizer.step()

                scheduler.step()

                if diagnostic_this_step:
                    first_delta_l2, first_delta_max = parameter_change_from_snapshot(first_update_before)
                    print(f"Gradient norm returned by clip: {float(clipped_norm):.12e}")
                    print(f"Parameter delta L2           : {first_delta_l2:.12e}")
                    print(f"Parameter delta max abs      : {first_delta_max:.12e}")

                    if first_delta_l2 <= 0.0 or first_delta_max <= 0.0:
                        raise RuntimeError("LoReFT parameters did not change on the first optimizer step with LR > 0.")

                    print("SUCCESS: LoReFT parameters changed on the first non-zero-LR optimizer step.")
                    print("=" * 100)
                    print_effective_rotation_norms("POST-FIRST-NONZERO-LR EFFECTIVE ROTATION-NORM CHECK")
                    first_nonzero_lr_update_checked = True

                optimizer.zero_grad(set_to_none=True)
                accumulation_count = 0

            current_lr = scheduler.get_last_lr()[0]
            gpu_gb = torch.cuda.memory_allocated() / 1024**3
            progress.set_postfix(loss=f"{current_loss:.4f}", lr=f"{current_lr:.2e}", gpu=f"{gpu_gb:.2f}GB")

            del batch
            del outputs
            del raw_loss
            del scaled_loss

        except torch.cuda.OutOfMemoryError as error:
            torch.cuda.empty_cache()
            raise RuntimeError(
                "\nCUDA OUT OF MEMORY DURING LoReFT R4.\n"
                "Do not switch this run to 4-bit, because that would make it a QLoRA experiment."
            ) from error

    mean_training_loss = float(np.mean(epoch_losses))
    epoch_parameter_delta_l2, epoch_parameter_delta_max = parameter_change_from_snapshot(epoch_parameter_snapshot)

    print("\nLoReFT epoch parameter change:")
    print(f"  L2 delta      : {epoch_parameter_delta_l2:.12e}")
    print(f"  Max abs delta : {epoch_parameter_delta_max:.12e}")

    if epoch_parameter_delta_l2 <= 0.0 or epoch_parameter_delta_max <= 0.0:
        raise RuntimeError(f"LoReFT parameters did not change during epoch {epoch}.")

    print_effective_rotation_norms(f"EPOCH {epoch} EFFECTIVE LoReFT ROTATION-NORM CHECK")

    print("\nCalculating validation loss...")
    arm_all_precision_diagnostics(loreft_interventions)
    mean_validation_loss = calculate_validation_loss()
    print_all_precision_diagnostics(
        loreft_interventions,
        f"EPOCH {epoch} FIRST-VALIDATION-BATCH TRUE-FP32 LoReFT DIAGNOSTIC",
    )

    print("\nRunning validation classification...")
    validation_metrics = evaluate_validation_generation(epoch)
    validation_ba = validation_metrics["Balanced_Accuracy"]
    epoch_minutes = (time.time() - epoch_start_time) / 60.0

    print("\n" + "-" * 100)
    print(f"R4 EPOCH {epoch} RESULTS")
    print("-" * 100)
    print(f"Training loss       : {mean_training_loss:.6f}")
    print(f"Validation loss     : {mean_validation_loss:.6f}")
    print(f"TN                  : {validation_metrics['TN']}")
    print(f"FP                  : {validation_metrics['FP']}")
    print(f"FN                  : {validation_metrics['FN']}")
    print(f"TP                  : {validation_metrics['TP']}")
    print(f"Sensitivity         : {validation_metrics['Sensitivity']:.4f}")
    print(f"Specificity         : {validation_metrics['Specificity']:.4f}")
    print(f"Balanced Accuracy   : {validation_metrics['Balanced_Accuracy']:.4f}")
    print(f"Accuracy            : {validation_metrics['Accuracy']:.4f}")
    print(f"F1                  : {validation_metrics['F1']:.4f}")

    training_history.append(
        {
            "epoch": epoch,
            "training_loss": mean_training_loss,
            "validation_loss": mean_validation_loss,
            "TN": validation_metrics["TN"],
            "FP": validation_metrics["FP"],
            "FN": validation_metrics["FN"],
            "TP": validation_metrics["TP"],
            "sensitivity": validation_metrics["Sensitivity"],
            "specificity": validation_metrics["Specificity"],
            "balanced_accuracy": validation_metrics["Balanced_Accuracy"],
            "accuracy": validation_metrics["Accuracy"],
            "precision": validation_metrics["Precision"],
            "f1": validation_metrics["F1"],
            "invalid_predictions": validation_metrics["Invalid_Predictions"],
            "epoch_minutes": epoch_minutes,
            "loreft_parameter_delta_l2": epoch_parameter_delta_l2,
            "loreft_parameter_delta_max_abs": epoch_parameter_delta_max,
        }
    )

    pd.DataFrame(training_history).to_csv(TRAINING_HISTORY_CSV, index=False)

    is_better = False
    if validation_ba > best_validation_balanced_accuracy:
        is_better = True
    elif (
        math.isclose(validation_ba, best_validation_balanced_accuracy, rel_tol=0.0, abs_tol=1e-12)
        and mean_validation_loss < best_validation_loss
    ):
        is_better = True

    if is_better:
        best_validation_balanced_accuracy = validation_ba
        best_validation_loss = mean_validation_loss
        best_epoch = epoch

        BEST_ADAPTER_DIR = os.path.join(BEST_CHECKPOINT_ROOT, f"e{epoch:03d}")
        os.makedirs(BEST_ADAPTER_DIR, exist_ok=False)

        BEST_REFT_WEIGHTS = os.path.join(BEST_ADAPTER_DIR, "loreft_intervention.pt")
        BEST_REFT_CONFIG = os.path.join(BEST_ADAPTER_DIR, "loreft_config.json")

        model.save_pretrained(BEST_ADAPTER_DIR, safe_serialization=True)

        print("\n" + "*" * 100)
        print("NEW BEST POOLED A2D2 LoReFT CHECKPOINT SAVED")
        print("*" * 100)
        print(f"Best epoch        : {best_epoch}")
        print(f"Validation BA     : {best_validation_balanced_accuracy:.4f}")
        print(f"Validation loss   : {best_validation_loss:.6f}")
        print(f"Early stopping    : 0/{EARLY_STOPPING_PATIENCE}")

        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        print("\n" + "-" * 100)
        print(
            f"NO NEW BEST CHECKPOINT | "
            f"Early stopping patience: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}"
        )
        print("-" * 100)

    gc.collect()
    torch.cuda.empty_cache()

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        stopped_early = True
        early_stop_epoch = epoch

        print("\n" + "=" * 100)
        print("EARLY STOPPING TRIGGERED")
        print("=" * 100)
        print(f"\nNo new best checkpoint for {EARLY_STOPPING_PATIENCE} consecutive epochs.")
        print(f"Stopping at epoch     : {early_stop_epoch}")
        print(f"Best epoch            : {best_epoch}")
        print(f"Best validation BA    : {best_validation_balanced_accuracy:.4f}")
        print(f"Best validation loss  : {best_validation_loss:.6f}")
        print("Final blinded tests will use the BEST saved checkpoint.")
        break


# %%
# ==================================================================================================
# 16. TRAINING COMPLETE + RELEASE TRAINING MODEL
# ==================================================================================================

total_training_minutes = (time.time() - training_start_time) / 60.0
peak_training_memory_gb = torch.cuda.max_memory_allocated(DEVICE) / 1024**3

print("\n" + "=" * 100)
print("POOLED A2D2 LoReFT R4 TRAINING COMPLETE")
print("=" * 100)
print(f"\nBest epoch          : {best_epoch}")
print(f"Best validation BA  : {best_validation_balanced_accuracy:.4f}")
print(f"Best validation loss: {best_validation_loss:.6f}")
print(f"Stopped early       : {stopped_early}")
if stopped_early:
    print(f"Stopped at epoch    : {early_stop_epoch}")
print(f"Training time       : {total_training_minutes:.2f} minutes")
print(f"Peak GPU allocated  : {peak_training_memory_gb:.2f} GB")

if best_epoch is None or BEST_ADAPTER_DIR is None:
    raise RuntimeError("No best checkpoint was saved.")

BEST_REFT_WEIGHTS = os.path.join(BEST_ADAPTER_DIR, "loreft_intervention.pt")
BEST_REFT_CONFIG = os.path.join(BEST_ADAPTER_DIR, "loreft_config.json")

if not os.path.isfile(BEST_REFT_WEIGHTS):
    raise FileNotFoundError(f"Best LoReFT weights were not found: {BEST_REFT_WEIGHTS}")
if not os.path.isfile(BEST_REFT_CONFIG):
    raise FileNotFoundError(f"Best LoReFT config was not found: {BEST_REFT_CONFIG}")

for hook_handle in loreft_hook_handles:
    try:
        hook_handle.remove()
    except Exception:
        pass

del model
del base_model
del optimizer
del scheduler
del trainable_parameters
del loreft_info
del target_language_layers
del loreft_hook_handles
del loreft_interventions
del loreft_controller

gc.collect()
torch.cuda.empty_cache()


# %%
# ==================================================================================================
# 17. LOAD FRESH MEDGEMMA + BEST LoReFT CHECKPOINT
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING FRESH MEDGEMMA FOR FINAL BLINDED TESTS")
print("=" * 100)

base_model_for_test = load_medgemma_base()

if hasattr(base_model_for_test.config, "use_cache"):
    base_model_for_test.config.use_cache = True

for parameter in base_model_for_test.parameters():
    parameter.requires_grad = False

with open(BEST_REFT_CONFIG, "r", encoding="utf-8") as file:
    best_reft_config = json.load(file)

try:
    saved_reft_state = torch.load(BEST_REFT_WEIGHTS, map_location=DEVICE, weights_only=True)
except TypeError:
    saved_reft_state = torch.load(BEST_REFT_WEIGHTS, map_location=DEVICE)

if not isinstance(saved_reft_state, dict):
    raise RuntimeError("Saved multi-layer LoReFT state is not a dictionary.")

model, test_loreft_info, test_loreft_controller, test_loreft_interventions, test_loreft_hook_handles, test_target_layers, _ = attach_loreft(
    base=base_model_for_test,
    target_layers_human=[int(x) for x in best_reft_config["target_layers_human"]],
    rank=int(best_reft_config["rank"]),
    prefix_tokens=int(best_reft_config["prefix_prompt_tokens"]),
    suffix_tokens=int(best_reft_config["suffix_prompt_tokens"]),
    dropout=float(best_reft_config["dropout"]),
    trainable=False,
    saved_state=saved_reft_state,
    saved_config=best_reft_config,
)

model.eval()

if test_loreft_info["num_language_layers"] != int(best_reft_config["num_language_layers"]):
    raise RuntimeError("Language-layer count differs between training and final test.")

print(f"\nBest LoReFT R4 checkpoint loaded from epoch {best_epoch}.")
print(
    f"Target layers: human {best_reft_config['target_layers_human']} / "
    f"{best_reft_config['num_language_layers']}"
)


# %%
# ==================================================================================================
# 18. FINAL BLINDED TEST LOOP
# ==================================================================================================

@torch.no_grad()
def evaluate_blinded_patient(patient_id, testing_df):
    print("\n" + "=" * 100)
    print(f"STARTING PATIENT {patient_id} A2D2 BLINDED TEST")
    print("=" * 100)

    patient_output_dir = os.path.join(TEST_OUTPUT_ROOT, f"patient_{patient_id}")
    os.makedirs(patient_output_dir, exist_ok=True)

    predictions_csv = os.path.join(patient_output_dir, f"patient_{patient_id}_predictions.csv")
    metrics_json = os.path.join(patient_output_dir, f"patient_{patient_id}_metrics.json")
    confusion_csv = os.path.join(patient_output_dir, f"patient_{patient_id}_confusion.csv")

    rows = []
    true_labels = []
    predictions = []
    test_start_time = time.time()

    progress = tqdm(range(len(testing_df)), desc=f"Patient {patient_id} blinded test")
    for index in progress:
        row = testing_df.iloc[index]
        image_path = str(row["_image_path"])

        raw_output, prediction = predict_single_image(image_path)
        true_label = int(row["_label"])

        true_labels.append(true_label)
        predictions.append(prediction)

        rows.append(
            {
                "patient_id": patient_id,
                "test_index": index,
                "image_path": image_path,
                "true_label": true_label,
                "true_class": "Seizure" if true_label == 1 else "Non-seizure",
                "raw_model_output": raw_output,
                "normalized_prediction": prediction if prediction is not None else "INVALID",
                "predicted_class": (
                    "Seizure"
                    if prediction == 1
                    else "Non-seizure"
                    if prediction == 0
                    else "INVALID"
                ),
            }
        )

        if (index + 1) % 25 == 0 or (index + 1) == len(testing_df):
            pd.DataFrame(rows).to_csv(predictions_csv, index=False)

    final_metrics, metric_predictions = calculate_binary_metrics(true_labels, predictions)

    prediction_df = pd.DataFrame(rows)
    prediction_df["metric_prediction"] = metric_predictions
    prediction_df["correct"] = prediction_df["true_label"].to_numpy() == metric_predictions

    original_metadata = testing_df.drop(columns=["_image_path", "_label"], errors="ignore").add_prefix("metadata_")
    prediction_df = pd.concat(
        [
            prediction_df.reset_index(drop=True),
            original_metadata.reset_index(drop=True),
        ],
        axis=1,
    )
    prediction_df.to_csv(predictions_csv, index=False)

    cm_df = pd.DataFrame(
        [
            [final_metrics["TN"], final_metrics["FP"]],
            [final_metrics["FN"], final_metrics["TP"]],
        ],
        index=["Actual_Non_Seizure", "Actual_Seizure"],
        columns=["Predicted_Non_Seizure", "Predicted_Seizure"],
    )
    cm_df.to_csv(confusion_csv)

    final_test_minutes = (time.time() - test_start_time) / 60.0

    final_metrics.update(
        {
            "Experiment": "Pooled_A2D2_R4",
            "Patient": patient_id,
            "Base_Model": MODEL_ID,
            "Representation": "A2D2-coefficient fixed-scale EEG variance topomap",
            "Method": "LoReFT",
            "Quantization": "None",
            "LoReFT_Rank": LOREFT_RANK,
            "LoReFT_Target_Layers_Human": LOREFT_TARGET_LAYERS_HUMAN,
            "LoReFT_Target_Layer_Python_Indices": LOREFT_TARGET_LAYER_INDICES,
            "LoReFT_Number_Interventions": len(LOREFT_TARGET_LAYER_INDICES),
            "LoReFT_Prefix_Prompt_Tokens": LOREFT_PREFIX_TOKENS,
            "LoReFT_Suffix_Prompt_Tokens": LOREFT_SUFFIX_TOKENS,
            "LoReFT_Dropout": LOREFT_DROPOUT,
            "Gradient_Checkpointing": False,
            "LoReFT_Train_Dtype": "float32",
            "Best_Checkpoint_Directory": BEST_ADAPTER_DIR,
            "Best_Epoch": best_epoch,
            "Best_Validation_Balanced_Accuracy": best_validation_balanced_accuracy,
            "Best_Validation_Loss": best_validation_loss,
            "Trainable_Parameters": trainable_parameter_count,
            "Total_Parameters": total_parameter_count,
            "Trainable_Percentage": trainable_percentage,
            "Training_Time_Minutes": total_training_minutes,
            "Final_Test_Time_Minutes": final_test_minutes,
            "Peak_Training_GPU_GB": peak_training_memory_gb,
            "Test_Metadata_CSV": TEST_METADATA_CSVS[patient_id],
            "Test_Total": len(testing_df),
            "Test_Seizure": int((testing_df["_label"] == 1).sum()),
            "Test_Non_Seizure": int((testing_df["_label"] == 0).sum()),
            "Predictions_CSV": predictions_csv,
            "Metrics_JSON": metrics_json,
            "Confusion_CSV": confusion_csv,
        }
    )

    with open(metrics_json, "w", encoding="utf-8") as file:
        json.dump(final_metrics, file, indent=4)

    print("\n" + "-" * 100)
    print(f"PATIENT {patient_id} FINAL RESULTS")
    print("-" * 100)
    print(f"Test images         : {len(testing_df)}")
    print(f"TN / FP / FN / TP   : {final_metrics['TN']} / {final_metrics['FP']} / {final_metrics['FN']} / {final_metrics['TP']}")
    print(f"Sensitivity         : {final_metrics['Sensitivity']:.4f}")
    print(f"Specificity         : {final_metrics['Specificity']:.4f}")
    print(f"Balanced Accuracy   : {final_metrics['Balanced_Accuracy']:.4f}")
    print(f"Accuracy            : {final_metrics['Accuracy']:.4f}")
    print(f"Precision           : {final_metrics['Precision']:.4f}")
    print(f"F1                  : {final_metrics['F1']:.4f}")
    print(f"Invalid outputs     : {final_metrics['Invalid_Predictions']}")
    print(f"Predictions CSV     : {predictions_csv}")
    print(f"Metrics JSON        : {metrics_json}")
    print(f"Confusion CSV       : {confusion_csv}")

    return final_metrics


all_patient_metrics = []

for patient_id in ["08", "11", "18", "24"]:
    patient_metrics = evaluate_blinded_patient(patient_id, testing_dfs[patient_id])
    all_patient_metrics.append(patient_metrics)
    gc.collect()
    torch.cuda.empty_cache()

all_metrics_df = pd.DataFrame(all_patient_metrics)
all_metrics_df.to_csv(ALL_PATIENT_METRICS_CSV, index=False)

with open(ALL_PATIENT_METRICS_JSON, "w", encoding="utf-8") as file:
    json.dump(all_patient_metrics, file, indent=4)


# %%
# ==================================================================================================
# 19. UPDATE CONFIG + FINAL REPORT
# ==================================================================================================

experiment_config["best_epoch"] = best_epoch
experiment_config["best_validation_balanced_accuracy"] = best_validation_balanced_accuracy
experiment_config["best_validation_loss"] = best_validation_loss
experiment_config["training_time_minutes"] = total_training_minutes
experiment_config["actual_epochs_run"] = len(training_history)
experiment_config["stopped_early"] = stopped_early
experiment_config["early_stop_epoch"] = early_stop_epoch
experiment_config["final_blinded_test_patients"] = ["08", "11", "18", "24"]
experiment_config["all_patient_metrics_csv"] = ALL_PATIENT_METRICS_CSV
experiment_config["all_patient_metrics_json"] = ALL_PATIENT_METRICS_JSON

with open(EXPERIMENT_CONFIG_JSON, "w", encoding="utf-8") as file:
    json.dump(experiment_config, file, indent=4)

print("\n" + "=" * 100)
print("POOLED A2D2 MEDGEMMA-4B LoReFT R4 COMPLETE")
print("=" * 100)

print("\nLoReFT R4 CONFIGURATION")
print(f"Rank                 : {LOREFT_RANK}")
print(f"Language layers      : {NUM_LANGUAGE_LAYERS}")
print(f"Target human layers  : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Target Python indices: {LOREFT_TARGET_LAYER_INDICES}")
print(f"Interventions        : {len(LOREFT_TARGET_LAYER_INDICES)}")
print(f"Prefix prompt tokens : {LOREFT_PREFIX_TOKENS}")
print(f"Suffix prompt tokens : {LOREFT_SUFFIX_TOKENS}")
print(f"Dropout              : {LOREFT_DROPOUT}")
print("Gradient checkpoint  : OFF")
print("LoReFT parameter dtype: float32")
print("LoReFT compute dtype : float32")
print("Initialization       : identity/no-op")
print("Rotation weight decay: 0.0")
print(f"Source weight decay  : {WEIGHT_DECAY}")
print(f"Learning rate        : {LEARNING_RATE}")
print(f"Epochs               : {NUM_EPOCHS}")
print(f"Early stop patience  : {EARLY_STOPPING_PATIENCE}")
print(f"Actual epochs run    : {len(training_history)}")
print(f"Stopped early        : {stopped_early}")

print("\nDATA")
print(
    f"Training             : {int((actual_train_df['_label'] == 1).sum())} seizure + "
    f"{int((actual_train_df['_label'] == 0).sum())} non-seizure = {len(actual_train_df)}"
)
print(
    f"Validation           : {int((validation_df['_label'] == 1).sum())} seizure + "
    f"{int((validation_df['_label'] == 0).sum())} non-seizure = {len(validation_df)}"
)

print("\nBEST CHECKPOINT")
print(f"Best epoch           : {best_epoch}")
print(f"Best validation BA   : {best_validation_balanced_accuracy:.4f}")
print(f"Best validation loss : {best_validation_loss:.6f}")
print(f"Best checkpoint      : {BEST_ADAPTER_DIR}")

print("\nPATIENT-WISE FINAL TEST SUMMARY")
summary_columns = [
    "Patient",
    "Test_Total",
    "TN",
    "FP",
    "FN",
    "TP",
    "Sensitivity",
    "Specificity",
    "Balanced_Accuracy",
    "Accuracy",
    "Precision",
    "F1",
    "Invalid_Predictions",
]
print(all_metrics_df[summary_columns].to_string(index=False))

print("\nFILES SAVED")
print("Training split       :", TRAIN_SPLIT_CSV)
print("Validation split     :", VALIDATION_SPLIT_CSV)
print("Training history     :", TRAINING_HISTORY_CSV)
print("Experiment config    :", EXPERIMENT_CONFIG_JSON)
print("Target info          :", TARGET_MODULES_TXT)
print("All metrics CSV      :", ALL_PATIENT_METRICS_CSV)
print("All metrics JSON     :", ALL_PATIENT_METRICS_JSON)
print("Per-patient outputs  :", TEST_OUTPUT_ROOT)
print("\n" + "=" * 100)


W1011 02:33:25.359000 30584 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels



HUGGING FACE LOGIN CHECK

Logged in as: sd377

GPU CHECK

GPU: NVIDIA GeForce RTX 5080
GPU memory: 15.92 GB
Model/training dtype: bfloat16

PACKAGE VERSIONS
Python       : 3.11.15
PyTorch      : 2.11.0+cu128
Transformers : 5.13.1

LOADING POOLED A2D2 TRAINING POOL

Rows: 2875

Columns:
   patient
   filename
   image_path
   class_folder
   label
   class_name
   global_serial
   global_serial_number
   window_serial
   window_serial_number

Image path column: image_path
Label column     : label

Class counts:
Seizure     : 575
Non-seizure : 2300
Total       : 2875

All 2875 image paths exist.

LOADING PATIENT 08 A2D2 BLINDED TEST

Rows: 1000

Columns:
   blinded_order
   blinded_id
   blinded_filename
   blinded_image_path
   global_serial
   true_label
   true_class

Image path column: blinded_image_path
Label column     : true_label

Class counts:
Seizure     : 200
Non-seizure : 800
Total       : 1000

All 1000 image paths exist.

LOADING PATIENT 11 A2D2 BLINDED TEST

Rows: 1000

C

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


SUCCESS: MedGemma-4B loaded.
GPU allocated: 8.02 GB
Gradient checkpointing disabled.
All pretrained MedGemma parameters frozen.


[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.



MEDGEMMA LANGUAGE STACK
Language layer path        : model.language_model.layers
Number of language layers  : 34
Language hidden size       : 2560
Intervention human layers  : [8, 17, 26, 31]
Intervention Python indices: [7, 16, 25, 30]
Number interventions       : 4

MULTI-LAYER LoReFT TRAINABLE PARAMETER REPORT
  base_model.model.language_model.layers.7._loreft_intervention.rotate_layer.parametrizations.weight.original: 10,240
  base_model.model.language_model.layers.7._loreft_intervention.learned_source.weight: 10,240
  base_model.model.language_model.layers.7._loreft_intervention.learned_source.bias: 4
  base_model.model.language_model.layers.16._loreft_intervention.rotate_layer.parametrizations.weight.original: 10,240
  base_model.model.language_model.layers.16._loreft_intervention.learned_source.weight: 10,240
  base_model.model.language_model.layers.16._loreft_intervention.learned_source.bias: 4
  base_model.model.language_model.layers.25._loreft_intervention.rotate_layer.param

... (61 outputs left out)


EPOCH 12 FIRST-VALIDATION-BATCH TRUE-FP32 LoReFT DIAGNOSTIC
Layer 8:
  Base hidden dtype             : torch.bfloat16
  LoReFT computation dtype      : torch.float32
  Raw FP32 delta L2 norm        : 2.243083789062e+04
  Effective post-cast delta norm: 2.245021875000e+04
  BF16 cast error L2 norm       : 1.945995635986e+02
  Retained delta-norm ratio     : 1.000864
  Hidden elements changed       : 99.9531%
  Raw FP32 mean |delta|         : 6.130369186401e+01
  Effective mean |delta|        : 6.130781936646e+01
Layer 17:
  Base hidden dtype             : torch.bfloat16
  LoReFT computation dtype      : torch.float32
  Raw FP32 delta L2 norm        : 3.461838281250e+04
  Effective post-cast delta norm: 3.461439843750e+04
  BF16 cast error L2 norm       : 9.562789916992e+01
  Retained delta-norm ratio     : 0.999885
  Hidden elements changed       : 99.8867%
  Raw FP32 mean |delta|         : 1.138035888672e+02
  Effective mean |delta|        : 1.138005218506e+02
Layer 26:
  Base hidden 

Validation predictions epoch 12: 100%|██████████| 575/575 [04:38<00:00,  2.06it/s]



----------------------------------------------------------------------------------------------------
R4 EPOCH 12 RESULTS
----------------------------------------------------------------------------------------------------
Training loss       : 0.038567
Validation loss     : 0.032235
TN                  : 447
FP                  : 13
FN                  : 13
TP                  : 102
Sensitivity         : 0.8870
Specificity         : 0.9717
Balanced Accuracy   : 0.9293
Accuracy            : 0.9548
F1                  : 0.8870

----------------------------------------------------------------------------------------------------
NO NEW BEST CHECKPOINT | Early stopping patience: 1/7
----------------------------------------------------------------------------------------------------

EPOCH 13/50


LoReFT R4 training epoch 13:   1%|          | 21/2300 [01:09<2:08:49,  3.39s/it, gpu=8.89GB, loss=0.0423, lr=7.20e-04]

In [1]:
# ==================================================================================================
# RESUME SETTINGS
# ==================================================================================================

# True = resume interrupted training
# False = start a fresh run
RESUME_TRAINING = True

# If you know the exact interrupted output folder, paste it here.
# If you do not know it, keep this empty and AUTO_RESUME_LATEST will find the latest checkpoint.
RESUME_OUTPUT_ROOT = r""

AUTO_RESUME_LATEST = True


def find_latest_resume_output_root(output_base):
    candidates = []

    if not os.path.isdir(output_base):
        return None

    for run_name in os.listdir(output_base):
        run_dir = os.path.join(output_base, run_name)
        state_path = os.path.join(
            run_dir,
            "latest_training_checkpoint",
            "training_state.pt",
        )

        if os.path.isfile(state_path):
            candidates.append((os.path.getmtime(state_path), run_dir))

    if not candidates:
        return None

    candidates = sorted(candidates, reverse=True)
    return candidates[0][1]


if RESUME_TRAINING:
    if RESUME_OUTPUT_ROOT.strip():
        OUTPUT_ROOT = os.path.normpath(
            os.path.expanduser(
                os.path.expandvars(
                    RESUME_OUTPUT_ROOT.strip().strip('"').strip("'")
                )
            )
        )
    elif AUTO_RESUME_LATEST:
        OUTPUT_ROOT = find_latest_resume_output_root(OUTPUT_BASE)
        if OUTPUT_ROOT is None:
            raise FileNotFoundError(
                "No resume checkpoint found under:\n"
                f"{OUTPUT_BASE}\n\n"
                "Your old run probably did not save latest_training_checkpoint/training_state.pt."
            )
    else:
        raise ValueError("Set RESUME_OUTPUT_ROOT or set AUTO_RESUME_LATEST=True.")

    RUN_ID = os.path.basename(OUTPUT_ROOT.rstrip(r"\/"))
    OUTPUT_BASE = os.path.dirname(OUTPUT_ROOT)

else:
    RUN_ID = time.strftime("%y%m%d_%H%M%S")
    OUTPUT_ROOT = os.path.join(OUTPUT_BASE, RUN_ID)


BEST_CHECKPOINT_ROOT = os.path.join(OUTPUT_ROOT, "ckpt")
TEST_OUTPUT_ROOT = os.path.join(OUTPUT_ROOT, "blinded_tests")
LATEST_CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, "latest_training_checkpoint")
LATEST_TRAINING_STATE = os.path.join(LATEST_CHECKPOINT_DIR, "training_state.pt")
LATEST_RESUME_INFO_JSON = os.path.join(LATEST_CHECKPOINT_DIR, "resume_info.json")

os.makedirs(BEST_CHECKPOINT_ROOT, exist_ok=True)
os.makedirs(TEST_OUTPUT_ROOT, exist_ok=True)
os.makedirs(LATEST_CHECKPOINT_DIR, exist_ok=True)

print("Resume training:", RESUME_TRAINING)
print("Output folder  :", OUTPUT_ROOT)
print("Resume state   :", LATEST_TRAINING_STATE)

if RESUME_TRAINING:
    if os.path.isfile(LATEST_TRAINING_STATE):
        print("Resume checkpoint found. Training will continue from the last completed epoch.")
    else:
        raise FileNotFoundError(f"Resume checkpoint not found:\n{LATEST_TRAINING_STATE}")

NameError: name 'OUTPUT_BASE' is not defined

In [2]:
# ==================================================================================================
# 1. PATHS + RESUME SETTINGS
# ==================================================================================================

TRAINING_METADATA_CSV = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\A2D2\Training\training_metadata.csv"
)

TEST_METADATA_CSVS = {
    "08": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patient 08\A2D2\Testing\Blinnded"
        r"\Patient_08_A2D2_blinded_test_labels.csv"
    ),
    "11": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patient 11\A2D2\TRAINING_TESTING_SPLIT_seed_99"
        r"\Testing\Blinded Topomaps\patient11_A2D2_blinded_labels.csv"
    ),
    "18": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patient 18\A2D2\Testing\Blinnded"
        r"\Patient_18_A2D2_blinded_test_labels.csv"
    ),
    "24": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patiient 24\A2D2\Testing\Blinnded"
        r"\Patient_24_A2D2_blinded_test_labels.csv"
    ),
}

OUTPUT_BASE = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4"
)

# True = resume interrupted run
# False = start fresh run
RESUME_TRAINING = True

# Option 1: paste exact interrupted output folder here.
# Option 2: keep empty and AUTO_RESUME_LATEST=True will find the latest checkpoint.
RESUME_OUTPUT_ROOT = r""

AUTO_RESUME_LATEST = True


def find_latest_resume_output_root(output_base):
    candidates = []

    if not os.path.isdir(output_base):
        return None

    for run_name in os.listdir(output_base):
        run_dir = os.path.join(output_base, run_name)
        state_path = os.path.join(
            run_dir,
            "latest_training_checkpoint",
            "training_state.pt",
        )

        if os.path.isfile(state_path):
            candidates.append((os.path.getmtime(state_path), run_dir))

    if not candidates:
        return None

    return sorted(candidates, reverse=True)[0][1]


if RESUME_TRAINING:
    if RESUME_OUTPUT_ROOT.strip():
        OUTPUT_ROOT = os.path.normpath(
            os.path.expanduser(
                os.path.expandvars(
                    RESUME_OUTPUT_ROOT.strip().strip('"').strip("'")
                )
            )
        )
    elif AUTO_RESUME_LATEST:
        OUTPUT_ROOT = find_latest_resume_output_root(OUTPUT_BASE)

        if OUTPUT_ROOT is None:
            raise FileNotFoundError(
                "No resume checkpoint found under:\n"
                f"{OUTPUT_BASE}\n\n"
                "That means the interrupted run did not create "
                "latest_training_checkpoint/training_state.pt."
            )
    else:
        raise ValueError("Set RESUME_OUTPUT_ROOT or set AUTO_RESUME_LATEST=True.")

    RUN_ID = os.path.basename(OUTPUT_ROOT.rstrip(r"\/"))
    OUTPUT_BASE = os.path.dirname(OUTPUT_ROOT)

else:
    RUN_ID = time.strftime("%y%m%d_%H%M%S")
    OUTPUT_ROOT = os.path.join(OUTPUT_BASE, RUN_ID)


BEST_CHECKPOINT_ROOT = os.path.join(OUTPUT_ROOT, "ckpt")
TEST_OUTPUT_ROOT = os.path.join(OUTPUT_ROOT, "blinded_tests")
LATEST_CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, "latest_training_checkpoint")
LATEST_TRAINING_STATE = os.path.join(LATEST_CHECKPOINT_DIR, "training_state.pt")
LATEST_RESUME_INFO_JSON = os.path.join(LATEST_CHECKPOINT_DIR, "resume_info.json")

os.makedirs(BEST_CHECKPOINT_ROOT, exist_ok=True)
os.makedirs(TEST_OUTPUT_ROOT, exist_ok=True)
os.makedirs(LATEST_CHECKPOINT_DIR, exist_ok=True)

print("Resume training:", RESUME_TRAINING)
print("Output folder  :", OUTPUT_ROOT)
print("Resume state   :", LATEST_TRAINING_STATE)

if RESUME_TRAINING:
    if os.path.isfile(LATEST_TRAINING_STATE):
        print("Resume checkpoint found. Training will continue from the last completed epoch.")
    else:
        raise FileNotFoundError(f"Resume checkpoint not found:\n{LATEST_TRAINING_STATE}")

NameError: name 'os' is not defined

In [3]:
import os
import time

OUTPUT_BASE = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4"
)

RESUME_TRAINING = True
RESUME_OUTPUT_ROOT = r""
AUTO_RESUME_LATEST = True


def find_latest_resume_output_root(output_base):
    candidates = []

    if not os.path.isdir(output_base):
        return None

    for run_name in os.listdir(output_base):
        run_dir = os.path.join(output_base, run_name)
        state_path = os.path.join(
            run_dir,
            "latest_training_checkpoint",
            "training_state.pt",
        )

        if os.path.isfile(state_path):
            candidates.append((os.path.getmtime(state_path), run_dir))

    if not candidates:
        return None

    return sorted(candidates, reverse=True)[0][1]


if RESUME_TRAINING:
    if RESUME_OUTPUT_ROOT.strip():
        OUTPUT_ROOT = os.path.normpath(
            os.path.expanduser(
                os.path.expandvars(
                    RESUME_OUTPUT_ROOT.strip().strip('"').strip("'")
                )
            )
        )
    elif AUTO_RESUME_LATEST:
        OUTPUT_ROOT = find_latest_resume_output_root(OUTPUT_BASE)

        if OUTPUT_ROOT is None:
            raise FileNotFoundError(
                "No resume checkpoint found under:\n"
                f"{OUTPUT_BASE}\n\n"
                "That means the interrupted run did not create "
                "latest_training_checkpoint/training_state.pt."
            )
    else:
        raise ValueError("Set RESUME_OUTPUT_ROOT or set AUTO_RESUME_LATEST=True.")

    RUN_ID = os.path.basename(OUTPUT_ROOT.rstrip(r"\/"))
    OUTPUT_BASE = os.path.dirname(OUTPUT_ROOT)

else:
    RUN_ID = time.strftime("%y%m%d_%H%M%S")
    OUTPUT_ROOT = os.path.join(OUTPUT_BASE, RUN_ID)


BEST_CHECKPOINT_ROOT = os.path.join(OUTPUT_ROOT, "ckpt")
TEST_OUTPUT_ROOT = os.path.join(OUTPUT_ROOT, "blinded_tests")
LATEST_CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, "latest_training_checkpoint")
LATEST_TRAINING_STATE = os.path.join(LATEST_CHECKPOINT_DIR, "training_state.pt")
LATEST_RESUME_INFO_JSON = os.path.join(LATEST_CHECKPOINT_DIR, "resume_info.json")

os.makedirs(BEST_CHECKPOINT_ROOT, exist_ok=True)
os.makedirs(TEST_OUTPUT_ROOT, exist_ok=True)
os.makedirs(LATEST_CHECKPOINT_DIR, exist_ok=True)

print("Resume training:", RESUME_TRAINING)
print("Output folder  :", OUTPUT_ROOT)
print("Resume state   :", LATEST_TRAINING_STATE)

if RESUME_TRAINING:
    if os.path.isfile(LATEST_TRAINING_STATE):
        print("Resume checkpoint found. Training will continue from the last completed epoch.")
    else:
        raise FileNotFoundError(f"Resume checkpoint not found:\n{LATEST_TRAINING_STATE}")

FileNotFoundError: No resume checkpoint found under:
C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4

That means the interrupted run did not create latest_training_checkpoint/training_state.pt.

In [4]:
import os
from pathlib import Path

OUTPUT_BASE = Path(
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4"
)

print("Checking:", OUTPUT_BASE)

for run_dir in sorted(OUTPUT_BASE.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True):
    if not run_dir.is_dir():
        continue

    resume_state = run_dir / "latest_training_checkpoint" / "training_state.pt"
    best_ckpts = list((run_dir / "ckpt").glob("e*/loreft_intervention.pt"))

    print("\nRUN:", run_dir)
    print("Exact resume checkpoint:", resume_state.exists())
    print("Best LoReFT checkpoints :", len(best_ckpts))

    for ckpt in best_ckpts:
        print("  ", ckpt)

Checking: C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4

RUN: C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_023326
Exact resume checkpoint: False
Best LoReFT checkpoints : 6
   C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_023326\ckpt\e001\loreft_intervention.pt
   C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_023326\ckpt\e002\loreft_intervention.pt
   C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_023326\ckpt\e003\loreft_intervention.pt
   C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_023326\ckpt\e007\loreft_intervention.pt
   C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_023326\ckpt\e008\loreft_intervention.pt
   C:\User

In [5]:
# ==================================================================================================
# FORCE EPOCH 11 AS BEST CHECKPOINT + VALIDATION + CONFUSION MATRIX
# ==================================================================================================

import os
import json
import torch
import pandas as pd

FORCED_BEST_EPOCH = 11

BEST_ADAPTER_DIR = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_023326\ckpt\e011"
)

BEST_REFT_WEIGHTS = os.path.join(BEST_ADAPTER_DIR, "loreft_intervention.pt")
BEST_REFT_CONFIG = os.path.join(BEST_ADAPTER_DIR, "loreft_config.json")

if not os.path.isfile(BEST_REFT_WEIGHTS):
    raise FileNotFoundError(f"LoReFT weights not found:\n{BEST_REFT_WEIGHTS}")

if not os.path.isfile(BEST_REFT_CONFIG):
    raise FileNotFoundError(f"LoReFT config not found:\n{BEST_REFT_CONFIG}")

print("=" * 100)
print("FORCING EPOCH 11 AS BEST LoReFT CHECKPOINT")
print("=" * 100)
print("Best checkpoint:", BEST_ADAPTER_DIR)

try:
    saved_reft_state = torch.load(BEST_REFT_WEIGHTS, map_location=DEVICE, weights_only=True)
except TypeError:
    saved_reft_state = torch.load(BEST_REFT_WEIGHTS, map_location=DEVICE)

for layer_key, intervention in loreft_interventions.items():
    if layer_key not in saved_reft_state:
        raise RuntimeError(f"Missing LoReFT layer {layer_key} in checkpoint.")

    intervention.load_state_dict(saved_reft_state[layer_key], strict=True)
    intervention.to(DEVICE)

best_epoch = FORCED_BEST_EPOCH

print("\nEpoch 11 LoReFT weights loaded successfully.")

print("\nCalculating validation loss for epoch 11...")
mean_validation_loss = calculate_validation_loss()

print("\nRunning validation classification for epoch 11...")
validation_metrics = evaluate_validation_generation(best_epoch)

best_validation_loss = mean_validation_loss
best_validation_balanced_accuracy = validation_metrics["Balanced_Accuracy"]

validation_confusion_df = pd.DataFrame(
    [
        [validation_metrics["TN"], validation_metrics["FP"]],
        [validation_metrics["FN"], validation_metrics["TP"]],
    ],
    index=["Actual_Non_Seizure", "Actual_Seizure"],
    columns=["Predicted_Non_Seizure", "Predicted_Seizure"],
)

forced_epoch_11_summary = {
    "forced_best_epoch": best_epoch,
    "best_adapter_dir": BEST_ADAPTER_DIR,
    "validation_loss": mean_validation_loss,
    "TN": validation_metrics["TN"],
    "FP": validation_metrics["FP"],
    "FN": validation_metrics["FN"],
    "TP": validation_metrics["TP"],
    "sensitivity": validation_metrics["Sensitivity"],
    "specificity": validation_metrics["Specificity"],
    "balanced_accuracy": validation_metrics["Balanced_Accuracy"],
    "accuracy": validation_metrics["Accuracy"],
    "precision": validation_metrics["Precision"],
    "f1": validation_metrics["F1"],
    "invalid_predictions": validation_metrics["Invalid_Predictions"],
}

forced_json = os.path.join(OUTPUT_ROOT, "forced_epoch_011_validation_metrics.json")
forced_csv = os.path.join(OUTPUT_ROOT, "forced_epoch_011_validation_metrics.csv")
forced_confusion_csv = os.path.join(OUTPUT_ROOT, "forced_epoch_011_validation_confusion_matrix.csv")

with open(forced_json, "w", encoding="utf-8") as f:
    json.dump(forced_epoch_11_summary, f, indent=4)

pd.DataFrame([forced_epoch_11_summary]).to_csv(forced_csv, index=False)
validation_confusion_df.to_csv(forced_confusion_csv)

print("\n" + "=" * 100)
print("FORCED EPOCH 11 VALIDATION RESULTS")
print("=" * 100)
print(f"Validation loss   : {mean_validation_loss:.6f}")
print(f"TN / FP / FN / TP : {validation_metrics['TN']} / {validation_metrics['FP']} / {validation_metrics['FN']} / {validation_metrics['TP']}")
print(f"Sensitivity       : {validation_metrics['Sensitivity']:.4f}")
print(f"Specificity       : {validation_metrics['Specificity']:.4f}")
print(f"Balanced Accuracy : {validation_metrics['Balanced_Accuracy']:.4f}")
print(f"Accuracy          : {validation_metrics['Accuracy']:.4f}")
print(f"Precision         : {validation_metrics['Precision']:.4f}")
print(f"F1                : {validation_metrics['F1']:.4f}")
print(f"Invalid outputs   : {validation_metrics['Invalid_Predictions']}")

print("\nValidation confusion matrix:")
display(validation_confusion_df)

print("\nSaved:")
print(forced_json)
print(forced_csv)
print(forced_confusion_csv)

FORCING EPOCH 11 AS BEST LoReFT CHECKPOINT
Best checkpoint: C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_023326\ckpt\e011


NameError: name 'DEVICE' is not defined

In [6]:
# ==================================================================================================
# FORCE EPOCH 11 AS BEST CHECKPOINT + VALIDATION + CONFUSION MATRIX
# Run this AFTER the model and LoReFT interventions are created.
# ==================================================================================================

import os
import json
import torch
import pandas as pd

if "DEVICE" not in globals():
    if torch.cuda.is_available():
        DEVICE = torch.device("cuda:0")
    else:
        DEVICE = torch.device("cpu")

required_names = [
    "loreft_interventions",
    "calculate_validation_loss",
    "evaluate_validation_generation",
    "OUTPUT_ROOT",
]

missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "You ran this cell too early. Run the main LoReFT code first until the model, "
        f"validation functions, and LoReFT interventions are created. Missing: {missing}"
    )

FORCED_BEST_EPOCH = 11

BEST_ADAPTER_DIR = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_023326\ckpt\e011"
)

BEST_REFT_WEIGHTS = os.path.join(BEST_ADAPTER_DIR, "loreft_intervention.pt")
BEST_REFT_CONFIG = os.path.join(BEST_ADAPTER_DIR, "loreft_config.json")

if not os.path.isfile(BEST_REFT_WEIGHTS):
    raise FileNotFoundError(f"LoReFT weights not found:\n{BEST_REFT_WEIGHTS}")

if not os.path.isfile(BEST_REFT_CONFIG):
    raise FileNotFoundError(f"LoReFT config not found:\n{BEST_REFT_CONFIG}")

print("=" * 100)
print("FORCING EPOCH 11 AS BEST LoReFT CHECKPOINT")
print("=" * 100)
print("Best checkpoint:", BEST_ADAPTER_DIR)

try:
    saved_reft_state = torch.load(BEST_REFT_WEIGHTS, map_location=DEVICE, weights_only=True)
except TypeError:
    saved_reft_state = torch.load(BEST_REFT_WEIGHTS, map_location=DEVICE)

for layer_key, intervention in loreft_interventions.items():
    if layer_key not in saved_reft_state:
        raise RuntimeError(f"Missing LoReFT layer {layer_key} in checkpoint.")

    intervention.load_state_dict(saved_reft_state[layer_key], strict=True)
    intervention.to(DEVICE)

best_epoch = FORCED_BEST_EPOCH

print("\nEpoch 11 LoReFT weights loaded successfully.")

print("\nCalculating validation loss for epoch 11...")
mean_validation_loss = calculate_validation_loss()

print("\nRunning validation classification for epoch 11...")
validation_metrics = evaluate_validation_generation(best_epoch)

best_validation_loss = mean_validation_loss
best_validation_balanced_accuracy = validation_metrics["Balanced_Accuracy"]

validation_confusion_df = pd.DataFrame(
    [
        [validation_metrics["TN"], validation_metrics["FP"]],
        [validation_metrics["FN"], validation_metrics["TP"]],
    ],
    index=["Actual_Non_Seizure", "Actual_Seizure"],
    columns=["Predicted_Non_Seizure", "Predicted_Seizure"],
)

forced_epoch_11_summary = {
    "forced_best_epoch": best_epoch,
    "best_adapter_dir": BEST_ADAPTER_DIR,
    "validation_loss": mean_validation_loss,
    "TN": validation_metrics["TN"],
    "FP": validation_metrics["FP"],
    "FN": validation_metrics["FN"],
    "TP": validation_metrics["TP"],
    "sensitivity": validation_metrics["Sensitivity"],
    "specificity": validation_metrics["Specificity"],
    "balanced_accuracy": validation_metrics["Balanced_Accuracy"],
    "accuracy": validation_metrics["Accuracy"],
    "precision": validation_metrics["Precision"],
    "f1": validation_metrics["F1"],
    "invalid_predictions": validation_metrics["Invalid_Predictions"],
}

forced_json = os.path.join(OUTPUT_ROOT, "forced_epoch_011_validation_metrics.json")
forced_csv = os.path.join(OUTPUT_ROOT, "forced_epoch_011_validation_metrics.csv")
forced_confusion_csv = os.path.join(OUTPUT_ROOT, "forced_epoch_011_validation_confusion_matrix.csv")

with open(forced_json, "w", encoding="utf-8") as f:
    json.dump(forced_epoch_11_summary, f, indent=4)

pd.DataFrame([forced_epoch_11_summary]).to_csv(forced_csv, index=False)
validation_confusion_df.to_csv(forced_confusion_csv)

print("\n" + "=" * 100)
print("FORCED EPOCH 11 VALIDATION RESULTS")
print("=" * 100)
print(f"Validation loss   : {mean_validation_loss:.6f}")
print(f"TN / FP / FN / TP : {validation_metrics['TN']} / {validation_metrics['FP']} / {validation_metrics['FN']} / {validation_metrics['TP']}")
print(f"Sensitivity       : {validation_metrics['Sensitivity']:.4f}")
print(f"Specificity       : {validation_metrics['Specificity']:.4f}")
print(f"Balanced Accuracy : {validation_metrics['Balanced_Accuracy']:.4f}")
print(f"Accuracy          : {validation_metrics['Accuracy']:.4f}")
print(f"Precision         : {validation_metrics['Precision']:.4f}")
print(f"F1                : {validation_metrics['F1']:.4f}")
print(f"Invalid outputs   : {validation_metrics['Invalid_Predictions']}")

print("\nValidation confusion matrix:")
display(validation_confusion_df)

print("\nSaved:")
print(forced_json)
print(forced_csv)
print(forced_confusion_csv)

RuntimeError: You ran this cell too early. Run the main LoReFT code first until the model, validation functions, and LoReFT interventions are created. Missing: ['loreft_interventions', 'calculate_validation_loss', 'evaluate_validation_generation']

In [7]:
# %%
# ==================================================================================================
# POOLED A2D2-COEFFICIENT EEG VARIANCE TOPOMAP CLASSIFICATION
# MEDGEMMA-4B-IT + LoReFT R4
#
# Train/validation:
#   - Pooled A2D2 training CSV from Patients 08, 11, 18, and 24
#   - 2875 total topomaps
#   - Deterministic stratified 80% train / 20% validation split
#
# Final testing:
#   - Reload the best LoReFT checkpoint into a fresh MedGemma model
#   - Evaluate separately on blinded A2D2 test CSVs for Patients 08, 11, 18, and 24
#
# LoReFT settings follow the reference script:
#   - rank = 4
#   - language layers = [8, 17, 26, 31]
#   - prefix prompt tokens = 5
#   - suffix prompt tokens = 5
#   - dropout = 0.0
#   - LR = 9e-4
#   - epoch limit = 50
#   - early stopping patience = 7 on validation balanced accuracy
#   - gradient checkpointing OFF
#   - LoReFT parameters and LoReFT computation stay in true FP32
#
# Labels:
#   0 = non-seizure
#   1 = seizure
# ==================================================================================================

import gc
import json
import math
import os
import random
import re
import sys
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
)
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import transformers
from huggingface_hub import whoami
from transformers import AutoModelForMultimodalLM, AutoProcessor, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")


# %%
# ==================================================================================================
# 1. PATHS
# ==================================================================================================

TRAINING_METADATA_CSV = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\A2D2\Training\training_metadata.csv"
)

TEST_METADATA_CSVS = {
    "08": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patient 08\A2D2\Testing\Blinnded"
        r"\Patient_08_A2D2_blinded_test_labels.csv"
    ),
    "11": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patient 11\A2D2\TRAINING_TESTING_SPLIT_seed_99"
        r"\Testing\Blinded Topomaps\patient11_A2D2_blinded_labels.csv"
    ),
    "18": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patient 18\A2D2\Testing\Blinnded"
        r"\Patient_18_A2D2_blinded_test_labels.csv"
    ),
    "24": (
        r"C:\Users\ga_sd360\OneDrive - The University of Akron"
        r"\Desktop\Fine Tuning\Patiient 24\A2D2\Testing\Blinnded"
        r"\Patient_24_A2D2_blinded_test_labels.csv"
    ),
}

# Keep this output path short because OneDrive + deep checkpoint folders can hit Windows path limits.
OUTPUT_BASE = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4"
)

RUN_ID = time.strftime("%y%m%d_%H%M%S")
OUTPUT_ROOT = os.path.join(OUTPUT_BASE, RUN_ID)
BEST_CHECKPOINT_ROOT = os.path.join(OUTPUT_ROOT, "ckpt")
TEST_OUTPUT_ROOT = os.path.join(OUTPUT_ROOT, "blinded_tests")

os.makedirs(BEST_CHECKPOINT_ROOT, exist_ok=True)
os.makedirs(TEST_OUTPUT_ROOT, exist_ok=True)

TRAIN_SPLIT_CSV = os.path.join(OUTPUT_ROOT, "train.csv")
VALIDATION_SPLIT_CSV = os.path.join(OUTPUT_ROOT, "val.csv")
TRAINING_HISTORY_CSV = os.path.join(OUTPUT_ROOT, "history.csv")
EXPERIMENT_CONFIG_JSON = os.path.join(OUTPUT_ROOT, "config.json")
TARGET_MODULES_TXT = os.path.join(OUTPUT_ROOT, "target.txt")
ALL_PATIENT_METRICS_CSV = os.path.join(OUTPUT_ROOT, "all_patient_metrics.csv")
ALL_PATIENT_METRICS_JSON = os.path.join(OUTPUT_ROOT, "all_patient_metrics.json")

BEST_ADAPTER_DIR = None
BEST_REFT_WEIGHTS = None
BEST_REFT_CONFIG = None


# %%
# ==================================================================================================
# 2. MODEL, PROMPTS, RANDOM SEED
# ==================================================================================================

MODEL_ID = "google/medgemma-4b-it"

SYSTEM_PROMPT = """
You are a strict vision-language classifier for EEG seizure detection.
Classify each EEG topomap image into exactly one of two labels:
Seizure
Non-seizure
Return only the final label. Do not provide explanation, punctuation, confidence score, or extra text.
""".strip()

USER_PROMPT = """
You are an expert in classifying seizure and non-seizure EEG patterns from composite wavelet-coefficient variance topomaps.
The attached composite image represents one 2-second EEG window:
- Left panel: A2 approximation-coefficient variance topomap, representing approximately 0-32 Hz.
- Right panel: D2 detail-coefficient variance topomap, representing approximately 32-64 Hz.
The EEG was divided into non-overlapping 2-second windows, filtered using a fourth-order Butterworth 59-61 Hz band-stop filter, and decomposed using a level-2 Discrete Wavelet Transform. For each bipolar EEG channel, the A2 and D2 coefficient variances were calculated and spatially interpolated at the midpoint between the two electrodes to create the topomaps. Each panel uses its own fixed coefficient-specific color scale across the complete dataset. Therefore, evaluate the relative intensity and spatial distribution within each panel. Do not directly compare the same color between the A2 and D2 panels as the same numerical value. Yellow indicates relatively higher coefficient variance within the corresponding panel, while dark purple indicates relatively lower coefficient variance.
Internally evaluate the image in three steps:
1. Examine the A2 panel for the intensity, spatial extent, localization, symmetry, and organization of elevated variance.
2. Examine the D2 panel independently using the same characteristics.
3. Combine the evidence from both panels and classify any image as seizure if both of the topomaps in one image look like seizure.
Return only:
seizure
non-seizure
""".strip()

RANDOM_SEED = 42


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_all_seeds(RANDOM_SEED)


# %%
# ==================================================================================================
# 3. LoReFT + TRAINING SETTINGS
# ==================================================================================================

LOREFT_RANK = 4
LOREFT_TARGET_LAYERS_HUMAN = [8, 17, 26, 31]
NUM_LOREFT_INTERVENTIONS = len(LOREFT_TARGET_LAYERS_HUMAN)
LOREFT_PREFIX_TOKENS = 5
LOREFT_SUFFIX_TOKENS = 5
LOREFT_DROPOUT = 0.0
LOREFT_COMPONENT = "language transformer block output / residual representation"
LOREFT_MODULE_NAME = "_loreft_intervention"

NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 7
LEARNING_RATE = 9e-4
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 1
VALIDATION_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.05
MAX_NEW_TOKENS = 8
NUM_WORKERS = 0

EXPECTED_FULL_TRAIN_TOTAL = 2875
EXPECTED_ACTUAL_TRAIN_TOTAL = 2300
EXPECTED_VALIDATION_TOTAL = 575
TRAIN_FRACTION = 0.80


# %%
# ==================================================================================================
# 4. ENVIRONMENT CHECKS
# ==================================================================================================

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "=" * 100)
print("HUGGING FACE LOGIN CHECK")
print("=" * 100)

try:
    hf_info = whoami()
    print("\nLogged in as:", hf_info["name"])
except Exception as error:
    raise RuntimeError(
        "\nHugging Face authentication is not available.\n"
        "Run this once in a notebook cell first:\n"
        "from huggingface_hub import login\n"
        "login()"
    ) from error

print("\n" + "=" * 100)
print("GPU CHECK")
print("=" * 100)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

DEVICE = torch.device("cuda:0")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"\nGPU: {GPU_NAME}")
print(f"GPU memory: {GPU_MEMORY_GB:.2f} GB")

if torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
    PRECISION_NAME = "bfloat16"
else:
    MODEL_DTYPE = torch.float16
    PRECISION_NAME = "float16"

torch.backends.cuda.matmul.allow_tf32 = True

print(f"Model/training dtype: {PRECISION_NAME}")
print("\n" + "=" * 100)
print("PACKAGE VERSIONS")
print("=" * 100)
print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")


# %%
# ==================================================================================================
# 5. METADATA LOADING
# ==================================================================================================

def clean_path(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().strip('"').strip("'")
    value = os.path.expandvars(value)
    value = os.path.expanduser(value)
    return os.path.normpath(value)


def csv_path_spelling_variants(csv_path):
    candidates = [csv_path]

    replacements = [
        ("Patiient 24", "Patient 24"),
        ("Patient 24", "Patiient 24"),
        ("All training together RAw", "All training together RAW"),
        ("All training together RAW", "All training together RAw"),
        ("bLINDED", "Blinded"),
        ("Blinded", "bLINDED"),
        ("Blinnded", "Blinded"),
        ("Blinded", "Blinnded"),
        ("Blinded folder", "Blinded"),
        ("Blinded", "Blinded folder"),
        ("Blinded Topomaps", "Blinded"),
        ("Blinded", "Blinded Topomaps"),
        ("TRAINING_TESTING_SPLIT", "TRAINING_TESTING_SPLIT_seed_99"),
        ("TRAINING_TESTING_SPLIT_seed_99", "TRAINING_TESTING_SPLIT"),
    ]

    for old, new in replacements:
        if old in csv_path:
            candidates.append(csv_path.replace(old, new))

    return unique_keep_order(candidates)


def patient_id_from_dataset_name(dataset_name):
    match = re.search(r"patient\s+(\d+)", str(dataset_name), flags=re.IGNORECASE)
    if match is None:
        return None
    return match.group(1).lstrip("0") or "0"


def score_csv_candidate(csv_path, patient_id):
    basename = os.path.basename(csv_path).lower()
    score = 0

    if "blind" in basename:
        score += 20
    if "raw" in basename or "a1" in basename:
        score += 10
    if "label" in basename:
        score += 5

    if patient_id is not None:
        patient_tokens = {
            patient_id,
            patient_id.zfill(2),
            f"patient_{patient_id}",
            f"patient_{patient_id.zfill(2)}",
            f"patient{patient_id}",
            f"patient{patient_id.zfill(2)}",
        }
        if any(token.lower() in basename for token in patient_tokens):
            score += 50

    return score


def resolve_csv_path(csv_path, dataset_name):
    csv_path = clean_path(csv_path)

    for candidate in csv_path_spelling_variants(csv_path):
        if os.path.isfile(candidate):
            if candidate != csv_path:
                print("\nCSV path repaired:")
                print("Original:", csv_path)
                print("Using   :", candidate)
            return candidate

    patient_id = patient_id_from_dataset_name(dataset_name)

    possible_parent_dirs = []
    for candidate in csv_path_spelling_variants(csv_path):
        possible_parent_dirs.append(os.path.dirname(candidate))

    possible_csvs = []
    for parent_dir in unique_keep_order(possible_parent_dirs):
        if not os.path.isdir(parent_dir):
            continue

        for filename in os.listdir(parent_dir):
            if filename.lower().endswith(".csv"):
                possible_csvs.append(os.path.join(parent_dir, filename))

    possible_csvs = unique_keep_order(possible_csvs)

    if len(possible_csvs) == 1:
        print("\nCSV filename repaired from the containing folder:")
        print("Original:", csv_path)
        print("Using   :", possible_csvs[0])
        return possible_csvs[0]

    if len(possible_csvs) > 1:
        scored = sorted(
            [
                (score_csv_candidate(candidate, patient_id), candidate)
                for candidate in possible_csvs
            ],
            reverse=True,
        )

        best_score, best_candidate = scored[0]
        tied_best = [
            candidate
            for score, candidate in scored
            if score == best_score
        ]

        if best_score > 0 and len(tied_best) == 1:
            print("\nCSV filename repaired from the containing folder:")
            print("Original:", csv_path)
            print("Using   :", best_candidate)
            return best_candidate

        print("\nCSV does not exist, and multiple CSV files were found in the folder.")
        print("Please choose the correct one from these candidates:")
        for _, candidate in scored:
            print(candidate)

    raise FileNotFoundError(f"\nCSV does not exist:\n{csv_path}")


def unique_keep_order(values):
    seen = set()
    ordered = []

    for value in values:
        if value is None:
            continue

        value = str(value).strip()
        if not value:
            continue

        if value not in seen:
            seen.add(value)
            ordered.append(value)

    return ordered


def filename_candidates_from_row(row, image_column):
    candidates = []

    for column in row.index:
        lower = str(column).lower()
        if "filename" in lower or lower in {"file", "image_name", "image"}:
            value = clean_path(row[column])
            if value:
                candidates.append(os.path.basename(value))

    original_path = clean_path(row[image_column])
    if original_path:
        candidates.append(os.path.basename(original_path))

    return unique_keep_order(candidates)


def class_folder_candidates_from_row(row, normalized_label):
    candidates = []

    for column in row.index:
        lower = str(column).lower()
        if "class" in lower or "category" in lower:
            value = str(row[column]).strip()
            if value and value.lower() != "nan":
                candidates.append(value)
                candidates.append(value.replace("-", "_").replace(" ", "_"))
                candidates.append(value.replace("_", "-").replace(" ", "-"))
                candidates.append(value.replace("_", " ").replace("-", " "))

    if int(normalized_label) == 1:
        candidates.extend(["Seizure", "seizure", "SEIZURE"])
    else:
        candidates.extend(
            [
                "Non_Seizure",
                "Non-Seizure",
                "Non Seizure",
                "Non-seizure",
                "non_seizure",
                "non-seizure",
                "non seizure",
            ]
        )

    return unique_keep_order(candidates)


def resolve_image_path_from_metadata(row, csv_path, image_column, normalized_label):
    original_path = clean_path(row[image_column])
    if original_path and os.path.isfile(original_path):
        return original_path, False

    csv_dir = os.path.dirname(os.path.abspath(csv_path))
    candidate_roots = unique_keep_order(
        [
            csv_dir,
            os.path.dirname(csv_dir),
            os.path.join(csv_dir, "Blinded"),
            os.path.join(csv_dir, "bLINDED"),
            os.path.join(csv_dir, "Testing"),
            os.path.join(csv_dir, "Training"),
        ]
    )

    filenames = filename_candidates_from_row(row, image_column)
    class_folders = class_folder_candidates_from_row(row, normalized_label)

    candidate_paths = []

    for root in candidate_roots:
        for filename in filenames:
            candidate_paths.append(os.path.join(root, filename))
            for class_folder in class_folders:
                candidate_paths.append(os.path.join(root, class_folder, filename))

    for candidate_path in unique_keep_order(candidate_paths):
        candidate_path = os.path.normpath(candidate_path)
        if os.path.isfile(candidate_path):
            return candidate_path, True

    return original_path, False


def detect_image_path_column(df, dataset_type):
    if dataset_type == "test":
        candidates = [
            "blinded_image_path",
            "blinded_topomap_path",
            "blinded_path",
            "blinded_image_filename",
            "blinded_topomap_filename",
            "fine_tuning_image_path",
            "image_path",
            "image_filename",
            "image_file",
            "image_name",
            "topomap_filename",
            "topomap_file",
            "topomap_name",
            "selected_image_path",
            "original_image_path",
            "file_path",
            "filepath",
            "filename",
            "file_name",
            "path",
        ]
    else:
        candidates = [
            "fine_tuning_image_path",
            "image_path",
            "image_filename",
            "image_file",
            "image_name",
            "topomap_filename",
            "topomap_file",
            "topomap_name",
            "selected_image_path",
            "blinded_image_path",
            "blinded_topomap_path",
            "blinded_path",
            "blinded_image_filename",
            "blinded_topomap_filename",
            "original_image_path",
            "file_path",
            "filepath",
            "filename",
            "file_name",
            "path",
        ]

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:
        lower = str(column).lower()
        if "path" in lower and ("image" in lower or "topomap" in lower or "blinded" in lower):
            return column
        if "filename" in lower and ("image" in lower or "topomap" in lower or "blind" in lower):
            return column

    raise ValueError(f"Could not detect image path column. Columns: {list(df.columns)}")


def detect_label_column(df):
    candidates = [
        "standard_label",
        "true_label",
        "label",
        "Label",
        "target",
        "true_class",
        "class_label",
        "class_name",
        "fine_tuning_class",
        "class_folder",
        "class",
        "category",
    ]

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:
        lower = str(column).lower()
        if "label" in lower or lower == "class":
            return column

    return None


def normalize_ground_truth_label(value):
    if pd.isna(value):
        return None

    if isinstance(value, (int, np.integer)):
        if int(value) == 1:
            return 1
        if int(value) == 0:
            return 0

    if isinstance(value, (float, np.floating)):
        if float(value) == 1.0:
            return 1
        if float(value) == 0.0:
            return 0

    text = str(value).strip().lower()
    text = text.replace("_", " ").replace("–", "-")
    text = re.sub(r"\s+", " ", text)

    if text in {"0", "0.0", "non seizure", "non-seizure", "nonseizure", "normal", "interictal", "ns"}:
        return 0
    if "non" in text and "seizure" in text:
        return 0
    if text in {"1", "1.0", "seizure", "ictal", "sz"}:
        return 1
    if "seizure" in text:
        return 1

    return None


def label_inference_columns(df):
    priority_columns = []

    for column in df.columns:
        lower = str(column).lower()

        if any(token in lower for token in ["label", "target", "category"]):
            priority_columns.append(column)
            continue

        if "class" in lower and "position" not in lower:
            priority_columns.append(column)
            continue

    for column in df.columns:
        lower = str(column).lower()

        if lower in {"filename", "file_name", "image_filename", "blinded_filename"}:
            priority_columns.append(column)
            continue

        if (
            ("filename" in lower or "path" in lower)
            and
            ("image" in lower or "topomap" in lower or "blind" in lower or "original" in lower)
        ):
            priority_columns.append(column)

    return unique_keep_order(priority_columns)


def infer_ground_truth_label_from_row(row, columns):
    for column in columns:
        label = normalize_ground_truth_label(row[column])
        if label is not None:
            return label

    return None


def load_and_standardize_metadata(csv_path, dataset_name, dataset_type):
    print("\n" + "=" * 100)
    print(f"LOADING {dataset_name.upper()}")
    print("=" * 100)

    csv_path = resolve_csv_path(csv_path, dataset_name)

    df = pd.read_csv(csv_path)
    print(f"\nRows: {len(df)}")
    print("\nColumns:")
    for column in df.columns:
        print("  ", column)

    image_column = detect_image_path_column(df, dataset_type)
    label_column = detect_label_column(df)
    print(f"\nImage path column: {image_column}")

    df["_image_path_original"] = df[image_column].apply(clean_path)

    if label_column is None:
        inference_columns = label_inference_columns(df)

        print("Label column     : inferred from metadata")
        print("Label inference columns:")
        for column in inference_columns:
            print("  ", column)

        if not inference_columns:
            raise ValueError(
                "Could not detect or infer a label column. "
                f"Columns: {list(df.columns)}"
            )

        df["_label"] = df.apply(
            lambda row: infer_ground_truth_label_from_row(row, inference_columns),
            axis=1
        )
    else:
        print(f"Label column     : {label_column}")
        df["_label"] = df[label_column].apply(normalize_ground_truth_label)

    invalid_labels = df[df["_label"].isna()]
    if len(invalid_labels) > 0:
        if label_column is None:
            preview_columns = label_inference_columns(df)
            print(invalid_labels[preview_columns].head(20))
        else:
            print(invalid_labels[[label_column]].drop_duplicates())
        raise ValueError("Some labels could not be interpreted.")

    df["_label"] = df["_label"].astype(int)

    resolved_paths = []
    repaired_count = 0

    for _, row in df.iterrows():
        resolved_path, repaired = resolve_image_path_from_metadata(
            row=row,
            csv_path=csv_path,
            image_column=image_column,
            normalized_label=int(row["_label"]),
        )
        resolved_paths.append(resolved_path)
        if repaired:
            repaired_count += 1

    df["_image_path"] = resolved_paths

    if repaired_count > 0:
        print(
            f"\nRepaired {repaired_count} image paths from the CSV folder "
            "using filename/class metadata."
        )

    missing_images = []
    for image_path in df["_image_path"]:
        if not os.path.isfile(image_path):
            missing_images.append(image_path)

    if missing_images:
        print(f"\nMissing images: {len(missing_images)}")
        print("\nThe CSV was found, but these image paths still do not exist.")
        print("Most likely the image_path column points to an old folder.")
        print("CSV folder used for automatic repair:")
        print(os.path.dirname(os.path.abspath(csv_path)))
        print("\nFirst missing paths:")
        for image_path in missing_images[:20]:
            print(image_path)
        raise FileNotFoundError("One or more images are missing.")

    duplicate_count = int(df["_image_path"].duplicated().sum())
    if duplicate_count > 0:
        raise ValueError(f"{duplicate_count} duplicate image paths detected in {dataset_name}.")

    seizure_count = int((df["_label"] == 1).sum())
    non_seizure_count = int((df["_label"] == 0).sum())

    print("\nClass counts:")
    print(f"Seizure     : {seizure_count}")
    print(f"Non-seizure : {non_seizure_count}")
    print(f"Total       : {len(df)}")
    print(f"\nAll {len(df)} image paths exist.")

    return df, image_column, label_column


full_training_df, train_image_column, train_label_column = load_and_standardize_metadata(
    TRAINING_METADATA_CSV,
    "pooled A2D2 training pool",
    "train",
)

if len(full_training_df) != EXPECTED_FULL_TRAIN_TOTAL:
    raise AssertionError(
        f"The pooled training CSV should contain {EXPECTED_FULL_TRAIN_TOTAL} rows, "
        f"but found {len(full_training_df)}."
    )

testing_dfs = {}
testing_column_info = {}

for patient_id, csv_path in TEST_METADATA_CSVS.items():
    test_df, image_column, label_column = load_and_standardize_metadata(
        csv_path,
        f"Patient {patient_id} A2D2 blinded test",
        "test",
    )
    testing_dfs[patient_id] = test_df
    testing_column_info[patient_id] = {
        "csv_path": csv_path,
        "image_column": image_column,
        "label_column": label_column,
    }


# %%
# ==================================================================================================
# 6. TRAIN / VALIDATION SPLIT + LEAKAGE CHECKS
# ==================================================================================================

seizure_pool = full_training_df[full_training_df["_label"] == 1].copy()
non_seizure_pool = full_training_df[full_training_df["_label"] == 0].copy()

seizure_pool = seizure_pool.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
non_seizure_pool = non_seizure_pool.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

train_seizure_count = int(round(len(seizure_pool) * TRAIN_FRACTION))
train_non_seizure_count = int(round(len(non_seizure_pool) * TRAIN_FRACTION))
validation_seizure_count = len(seizure_pool) - train_seizure_count
validation_non_seizure_count = len(non_seizure_pool) - train_non_seizure_count

actual_train_df = pd.concat(
    [
        seizure_pool.iloc[:train_seizure_count].copy(),
        non_seizure_pool.iloc[:train_non_seizure_count].copy(),
    ],
    ignore_index=True,
)

validation_df = pd.concat(
    [
        seizure_pool.iloc[train_seizure_count:].copy(),
        non_seizure_pool.iloc[train_non_seizure_count:].copy(),
    ],
    ignore_index=True,
)

actual_train_df = actual_train_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
validation_df = validation_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

if len(actual_train_df) != EXPECTED_ACTUAL_TRAIN_TOTAL:
    raise AssertionError(f"Expected {EXPECTED_ACTUAL_TRAIN_TOTAL} training images, found {len(actual_train_df)}.")
if len(validation_df) != EXPECTED_VALIDATION_TOTAL:
    raise AssertionError(f"Expected {EXPECTED_VALIDATION_TOTAL} validation images, found {len(validation_df)}.")

train_validation_overlap = set(actual_train_df["_image_path"]).intersection(set(validation_df["_image_path"]))
if train_validation_overlap:
    raise RuntimeError(f"Train/validation leakage detected: {len(train_validation_overlap)} identical image paths.")

for patient_id, test_df in testing_dfs.items():
    train_test_overlap = set(full_training_df["_image_path"]).intersection(set(test_df["_image_path"]))
    if train_test_overlap:
        raise RuntimeError(
            f"Training/test leakage detected for Patient {patient_id}: "
            f"{len(train_test_overlap)} identical image paths."
        )

print("\n" + "=" * 100)
print("POOLED A2D2 TRAIN / VALIDATION SPLIT")
print("=" * 100)
print(
    f"\nTRAIN: {int((actual_train_df['_label'] == 1).sum())} seizure + "
    f"{int((actual_train_df['_label'] == 0).sum())} non-seizure = {len(actual_train_df)}"
)
print(
    f"VALIDATION: {int((validation_df['_label'] == 1).sum())} seizure + "
    f"{int((validation_df['_label'] == 0).sum())} non-seizure = {len(validation_df)}"
)
print("\nTraining/validation/test current image-path overlap: 0")

actual_train_df.to_csv(TRAIN_SPLIT_CSV, index=False)
validation_df.to_csv(VALIDATION_SPLIT_CSV, index=False)


# %%
# ==================================================================================================
# 7. DATASET, PROCESSOR, BASE MODEL
# ==================================================================================================

class EEGTopomapDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        return {
            "index": int(index),
            "image_path": str(row["_image_path"]),
            "label": int(row["_label"]),
        }


print("\n" + "=" * 100)
print("LOADING MEDGEMMA PROCESSOR")
print("=" * 100)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=True)
if hasattr(processor, "tokenizer"):
    processor.tokenizer.padding_side = "right"

print("\nProcessor loaded successfully.")


def load_medgemma_base():
    try:
        loaded_model = AutoModelForMultimodalLM.from_pretrained(
            MODEL_ID,
            token=True,
            dtype=MODEL_DTYPE,
            low_cpu_mem_usage=True,
            attn_implementation="sdpa",
        )
    except TypeError:
        loaded_model = AutoModelForMultimodalLM.from_pretrained(
            MODEL_ID,
            token=True,
            torch_dtype=MODEL_DTYPE,
            low_cpu_mem_usage=True,
            attn_implementation="sdpa",
        )
    return loaded_model.to(DEVICE)


print("\n" + "=" * 100)
print("LOADING MEDGEMMA-4B")
print("=" * 100)

gc.collect()
torch.cuda.empty_cache()

base_model = load_medgemma_base()
print("\nSUCCESS: MedGemma-4B loaded.")
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

if hasattr(base_model.config, "use_cache"):
    base_model.config.use_cache = False

for parameter in base_model.parameters():
    parameter.requires_grad = False

if hasattr(base_model, "gradient_checkpointing_disable"):
    try:
        base_model.gradient_checkpointing_disable()
    except Exception:
        pass

if hasattr(base_model.config, "gradient_checkpointing"):
    try:
        base_model.config.gradient_checkpointing = False
    except Exception:
        pass

if bool(getattr(base_model, "is_gradient_checkpointing", False)):
    raise RuntimeError("Gradient checkpointing is still active; LoReFT hook run requires it OFF.")

print("Gradient checkpointing disabled.")
print("All pretrained MedGemma parameters frozen.")


# %%
# ==================================================================================================
# 8. LoReFT IMPLEMENTATION
# ==================================================================================================

def find_language_layers(multimodal_model):
    candidates = []

    if hasattr(multimodal_model, "model"):
        outer = multimodal_model.model
        if hasattr(outer, "language_model"):
            candidates.append(("model.language_model", outer.language_model))
        if hasattr(outer, "text_model"):
            candidates.append(("model.text_model", outer.text_model))

    if hasattr(multimodal_model, "language_model"):
        candidates.append(("language_model", multimodal_model.language_model))

    if hasattr(multimodal_model, "text_model"):
        candidates.append(("text_model", multimodal_model.text_model))

    for base_path, language_model in candidates:
        if hasattr(language_model, "layers") and isinstance(language_model.layers, nn.ModuleList):
            return language_model, language_model.layers, f"{base_path}.layers"

        if hasattr(language_model, "model"):
            nested = language_model.model
            if hasattr(nested, "layers") and isinstance(nested.layers, nn.ModuleList):
                return language_model, nested.layers, f"{base_path}.model.layers"

    for module_name, module in multimodal_model.named_modules():
        if (
            isinstance(module, nn.ModuleList)
            and module_name.endswith("layers")
            and "language_model" in module_name.lower()
            and "vision" not in module_name.lower()
        ):
            return None, module, module_name

    raise RuntimeError("Could not locate MedGemma's language transformer stack.")


class LowRankRotateLayer(nn.Module):
    def __init__(self, input_dim, rank):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(input_dim, rank), requires_grad=True)
        nn.init.orthogonal_(self.weight)

    def forward(self, x):
        return torch.matmul(x.to(self.weight.dtype), self.weight)


class LoReFTIntervention(nn.Module):
    def __init__(self, embed_dim, rank, train_dtype=torch.float32, dropout=0.0):
        super().__init__()

        rotate_layer = LowRankRotateLayer(embed_dim, rank).to(dtype=train_dtype)
        self.rotate_layer = nn.utils.parametrizations.orthogonal(rotate_layer)
        self.learned_source = nn.Linear(embed_dim, rank, bias=True).to(dtype=train_dtype)
        self.dropout = nn.Dropout(dropout)

        with torch.no_grad():
            current_rotation = self.rotate_layer.weight.detach().to(
                dtype=self.learned_source.weight.dtype,
                device=self.learned_source.weight.device,
            )
            self.learned_source.weight.copy_(current_rotation.T)
            self.learned_source.bias.zero_()

        self._capture_precision_diagnostic = False
        self._last_precision_diagnostic = None

    def arm_precision_diagnostic(self):
        self._capture_precision_diagnostic = True
        self._last_precision_diagnostic = None

    def get_precision_diagnostic(self):
        return self._last_precision_diagnostic

    def forward(self, base):
        original_dtype = base.dtype
        autocast_device_type = "cuda" if base.is_cuda else "cpu"

        with torch.autocast(device_type=autocast_device_type, enabled=False):
            base_fp32 = base.float()

            if self.rotate_layer.weight.dtype != torch.float32:
                raise RuntimeError(f"LoReFT rotate_layer weight is not FP32: {self.rotate_layer.weight.dtype}")
            if self.learned_source.weight.dtype != torch.float32:
                raise RuntimeError(f"LoReFT learned_source weight is not FP32: {self.learned_source.weight.dtype}")

            rotated_base_fp32 = self.rotate_layer(base_fp32)
            learned_source_fp32 = self.learned_source(base_fp32)
            delta_fp32 = torch.matmul(
                learned_source_fp32 - rotated_base_fp32,
                self.rotate_layer.weight.T,
            )
            updated_fp32 = base_fp32 + delta_fp32

        updated = updated_fp32.to(dtype=original_dtype)

        if self._capture_precision_diagnostic:
            with torch.no_grad():
                base_detached_fp32 = base.detach().float()
                raw_delta = delta_fp32.detach().float()
                updated_detached_fp32 = updated.detach().float()
                effective_delta = updated_detached_fp32 - base_detached_fp32

                raw_norm = float(raw_delta.norm().cpu())
                effective_norm = float(effective_delta.norm().cpu())
                cast_error_norm = float((raw_delta - effective_delta).norm().cpu())
                changed_fraction = float((updated.detach() != base.detach()).float().mean().cpu())
                retained_ratio = effective_norm / raw_norm if raw_norm > 0.0 else 0.0

                self._last_precision_diagnostic = {
                    "base_dtype": str(base.dtype),
                    "reft_dtype": str(delta_fp32.dtype),
                    "raw_fp32_delta_norm": raw_norm,
                    "effective_post_cast_delta_norm": effective_norm,
                    "cast_error_norm": cast_error_norm,
                    "retained_norm_ratio": retained_ratio,
                    "changed_element_fraction": changed_fraction,
                    "raw_fp32_delta_mean_abs": float(raw_delta.abs().mean().cpu()),
                    "effective_post_cast_delta_mean_abs": float(effective_delta.abs().mean().cpu()),
                }

            self._capture_precision_diagnostic = False

        return self.dropout(updated)


class LoReFTPositionController:
    def __init__(self, prefix_tokens, suffix_tokens):
        self.prefix_tokens = int(prefix_tokens)
        self.suffix_tokens = int(suffix_tokens)
        self.active = False
        self.prompt_length = None
        self.minimum_sequence_length = None

    def activate(self, prompt_length, minimum_sequence_length=None):
        self.prompt_length = int(prompt_length)
        self.minimum_sequence_length = int(minimum_sequence_length if minimum_sequence_length is not None else prompt_length)
        self.active = True

    def deactivate(self):
        self.active = False
        self.prompt_length = None
        self.minimum_sequence_length = None

    def get_positions(self, current_sequence_length):
        if not self.active:
            return []
        if current_sequence_length < self.minimum_sequence_length:
            return []

        prompt_length = min(self.prompt_length, int(current_sequence_length))
        prefix = list(range(min(self.prefix_tokens, prompt_length)))
        suffix = list(range(max(0, prompt_length - self.suffix_tokens), prompt_length))

        positions = []
        seen = set()
        for position in prefix + suffix:
            if position not in seen:
                seen.add(position)
                positions.append(position)

        return positions


def _extract_hidden(layer_output):
    if torch.is_tensor(layer_output):
        return layer_output
    if isinstance(layer_output, (tuple, list)) and len(layer_output) > 0 and torch.is_tensor(layer_output[0]):
        return layer_output[0]
    raise TypeError(f"Unsupported language-layer output type: {type(layer_output)}")


def _replace_hidden(layer_output, new_hidden):
    if torch.is_tensor(layer_output):
        return new_hidden
    if isinstance(layer_output, tuple):
        return (new_hidden,) + tuple(layer_output[1:])
    if isinstance(layer_output, list):
        return [new_hidden] + list(layer_output[1:])
    raise TypeError(f"Unsupported language-layer output type: {type(layer_output)}")


def make_loreft_hook(intervention, controller):
    def hook(module, inputs, output):
        hidden = _extract_hidden(output)

        if hidden.ndim != 3:
            raise RuntimeError(f"Expected [batch, sequence, hidden], got {tuple(hidden.shape)}")

        positions = controller.get_positions(hidden.shape[1])
        if not positions:
            return output

        position_tensor = torch.tensor(positions, dtype=torch.long, device=hidden.device)
        selected = hidden.index_select(1, position_tensor)
        modified = intervention(selected)

        new_hidden = hidden.clone()
        new_hidden[:, position_tensor, :] = modified
        return _replace_hidden(output, new_hidden)

    return hook


class MedGemmaLoReFTWrapper(nn.Module):
    def __init__(self, base, controller, interventions, checkpoint_info):
        super().__init__()
        self.base_model = base
        self.controller = controller
        self.checkpoint_info = dict(checkpoint_info)
        object.__setattr__(self, "_interventions_ref", interventions)

    @property
    def config(self):
        return self.base_model.config

    def forward(self, *args, **kwargs):
        labels = kwargs.get("labels", None)

        if labels is not None:
            if labels.ndim != 2 or labels.shape[0] != 1:
                raise ValueError("This LoReFT pipeline expects physical batch size = 1.")

            supervised_positions = torch.nonzero(labels[0] != -100, as_tuple=False).flatten()
            if supervised_positions.numel() == 0:
                raise RuntimeError("No assistant target token was found in the loss mask.")

            prompt_length = int(supervised_positions[0].item())
        else:
            input_ids = kwargs.get("input_ids", None)
            if input_ids is None:
                raise RuntimeError("LoReFT forward needs labels or input_ids to determine prompt positions.")
            prompt_length = int(input_ids.shape[1])

        self.controller.activate(prompt_length=prompt_length, minimum_sequence_length=prompt_length)
        return self.base_model(*args, **kwargs)

    @torch.no_grad()
    def generate(self, *args, **kwargs):
        input_ids = kwargs.get("input_ids", None)
        if input_ids is None:
            raise RuntimeError("generate() requires input_ids for LoReFT position control.")

        prompt_length = int(input_ids.shape[1])
        self.controller.activate(prompt_length=prompt_length, minimum_sequence_length=prompt_length)

        try:
            return self.base_model.generate(*args, **kwargs)
        finally:
            self.controller.deactivate()

    def save_pretrained(self, save_directory, safe_serialization=True, **kwargs):
        os.makedirs(save_directory, exist_ok=True)

        multi_layer_state = {
            layer_key: intervention.state_dict()
            for layer_key, intervention in self._interventions_ref.items()
        }

        torch.save(multi_layer_state, os.path.join(save_directory, "loreft_intervention.pt"))

        with open(os.path.join(save_directory, "loreft_config.json"), "w", encoding="utf-8") as file:
            json.dump(self.checkpoint_info, file, indent=4)


def resolve_loreft_model_info(multimodal_model, target_layers_human):
    language_model_object, language_layers, language_layers_path = find_language_layers(multimodal_model)
    num_language_layers = len(language_layers)

    target_layers_human = [int(x) for x in target_layers_human]
    if len(target_layers_human) != len(set(target_layers_human)):
        raise ValueError("Duplicate LoReFT target layers were requested.")

    for human_layer in target_layers_human:
        if human_layer < 1 or human_layer > num_language_layers:
            raise ValueError(f"Requested LoReFT layer {human_layer}, but the language model has {num_language_layers} layers.")

    target_layer_indices = [human_layer - 1 for human_layer in target_layers_human]

    if hasattr(multimodal_model.config, "text_config") and hasattr(multimodal_model.config.text_config, "hidden_size"):
        hidden_size = int(multimodal_model.config.text_config.hidden_size)
    elif language_model_object is not None and hasattr(language_model_object, "config"):
        hidden_size = int(language_model_object.config.hidden_size)
    else:
        raise RuntimeError("Could not determine the language hidden size.")

    return {
        "language_model_object": language_model_object,
        "language_layers": language_layers,
        "language_layers_path": language_layers_path,
        "num_language_layers": num_language_layers,
        "hidden_size": hidden_size,
        "target_layers_human": target_layers_human,
        "target_layer_indices": target_layer_indices,
    }


def attach_loreft(
    base,
    target_layers_human,
    rank,
    prefix_tokens,
    suffix_tokens,
    dropout,
    trainable,
    saved_state=None,
    saved_config=None,
):
    info = resolve_loreft_model_info(base, target_layers_human)
    language_layers = info["language_layers"]
    target_layer_indices = info["target_layer_indices"]

    controller = LoReFTPositionController(prefix_tokens=prefix_tokens, suffix_tokens=suffix_tokens)
    target_language_layers = []
    interventions = {}
    hook_handles = []

    for human_layer, python_index in zip(info["target_layers_human"], target_layer_indices):
        layer_key = str(human_layer)
        target_layer = language_layers[python_index]

        intervention = LoReFTIntervention(
            embed_dim=info["hidden_size"],
            rank=rank,
            train_dtype=torch.float32,
            dropout=dropout,
        ).to(DEVICE)

        target_layer.add_module(LOREFT_MODULE_NAME, intervention)

        if saved_state is not None:
            if layer_key not in saved_state:
                raise RuntimeError(f"Missing saved LoReFT state for human layer {human_layer}.")
            intervention.load_state_dict(saved_state[layer_key], strict=True)

        for parameter in intervention.parameters():
            parameter.requires_grad = bool(trainable)

        hook_handle = target_layer.register_forward_hook(make_loreft_hook(intervention, controller))

        target_language_layers.append(target_layer)
        interventions[layer_key] = intervention
        hook_handles.append(hook_handle)

    checkpoint_info = {
        "method": "LoReFT",
        "variant": "multi_layer_independent_interventions",
        "formula": "h + R^T(Wh + b - Rh)",
        "rank": rank,
        "num_language_layers": info["num_language_layers"],
        "target_layers_human": info["target_layers_human"],
        "target_layer_python_indices": target_layer_indices,
        "num_interventions": len(target_layer_indices),
        "hidden_size": info["hidden_size"],
        "prefix_prompt_tokens": prefix_tokens,
        "suffix_prompt_tokens": suffix_tokens,
        "dropout": dropout,
        "component": LOREFT_COMPONENT,
        "language_layers_path": info["language_layers_path"],
        "precision_fix": "autocast disabled inside LoReFT; true FP32 projection + FP32 residual addition; one final cast to model dtype",
        "initialization": "identity_no_op_W_equals_R_transpose_bias_zero",
        "bf16_diagnostic": True,
        "rotation_weight_decay": 0.0,
        "source_weight_decay": WEIGHT_DECAY,
        "source_bias_weight_decay": 0.0,
    }

    if saved_config is not None:
        checkpoint_info.update(saved_config)

    wrapper = MedGemmaLoReFTWrapper(
        base=base,
        controller=controller,
        interventions=interventions,
        checkpoint_info=checkpoint_info,
    ).to(DEVICE)

    return wrapper, info, controller, interventions, hook_handles, target_language_layers, checkpoint_info


model, loreft_info, loreft_controller, loreft_interventions, loreft_hook_handles, target_language_layers, checkpoint_info = attach_loreft(
    base=base_model,
    target_layers_human=LOREFT_TARGET_LAYERS_HUMAN,
    rank=LOREFT_RANK,
    prefix_tokens=LOREFT_PREFIX_TOKENS,
    suffix_tokens=LOREFT_SUFFIX_TOKENS,
    dropout=LOREFT_DROPOUT,
    trainable=True,
)

NUM_LANGUAGE_LAYERS = loreft_info["num_language_layers"]
LOREFT_HIDDEN_SIZE = loreft_info["hidden_size"]
LANGUAGE_LAYERS_PATH = loreft_info["language_layers_path"]
LOREFT_TARGET_LAYER_INDICES = loreft_info["target_layer_indices"]

print("\n" + "=" * 100)
print("MEDGEMMA LANGUAGE STACK")
print("=" * 100)
print(f"Language layer path        : {LANGUAGE_LAYERS_PATH}")
print(f"Number of language layers  : {NUM_LANGUAGE_LAYERS}")
print(f"Language hidden size       : {LOREFT_HIDDEN_SIZE}")
print(f"Intervention human layers  : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Intervention Python indices: {LOREFT_TARGET_LAYER_INDICES}")
print(f"Number interventions       : {len(LOREFT_TARGET_LAYER_INDICES)}")


# %%
# ==================================================================================================
# 9. TRAINABLE PARAMETER CHECKS
# ==================================================================================================

trainable_named_parameters = [
    (name, parameter)
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

unexpected_trainable = [
    name
    for name, parameter in trainable_named_parameters
    if LOREFT_MODULE_NAME not in name
]

if unexpected_trainable:
    raise RuntimeError("Unexpected pretrained parameters are trainable:\n" + "\n".join(unexpected_trainable[:50]))

if not trainable_named_parameters:
    raise RuntimeError("No LoReFT parameters are trainable.")

non_fp32_trainable = [
    f"{name}: {parameter.dtype}"
    for name, parameter in trainable_named_parameters
    if parameter.dtype != torch.float32
]

if non_fp32_trainable:
    raise RuntimeError(
        "Every trainable LoReFT parameter must remain FP32, but these do not:\n"
        + "\n".join(non_fp32_trainable)
    )

trainable_parameter_count = sum(parameter.numel() for name, parameter in trainable_named_parameters)
total_parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_percentage = 100.0 * trainable_parameter_count / total_parameter_count

expected_per_intervention = 2 * LOREFT_HIDDEN_SIZE * LOREFT_RANK + LOREFT_RANK
expected_total_loreft_parameters = expected_per_intervention * len(LOREFT_TARGET_LAYER_INDICES)

if trainable_parameter_count != expected_total_loreft_parameters:
    raise RuntimeError(
        "Unexpected multi-layer LoReFT trainable-parameter count.\n"
        f"Expected: {expected_total_loreft_parameters:,}\n"
        f"Found   : {trainable_parameter_count:,}"
    )

print("\n" + "=" * 100)
print("MULTI-LAYER LoReFT TRAINABLE PARAMETER REPORT")
print("=" * 100)
for name, parameter in trainable_named_parameters:
    print(f"  {name}: {parameter.numel():,}")

print(f"\nIntervention layers  : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Parameters / layer   : {expected_per_intervention:,}")
print(f"Trainable parameters : {trainable_parameter_count:,}")
print(f"Total parameters     : {total_parameter_count:,}")
print(f"Trainable percentage : {trainable_percentage:.6f}%")
print("LoReFT parameter dtype: torch.float32")
print("LoReFT compute dtype  : torch.float32")
print("Initialization        : identity/no-op (W = R^T, b = 0)")

with open(TARGET_MODULES_TXT, "w", encoding="utf-8") as file:
    for key, value in checkpoint_info.items():
        file.write(f"{key}={value}\n")


# %%
# ==================================================================================================
# 10. PROMPT, COLLATOR, LOADERS
# ==================================================================================================

def label_to_target_text(label):
    if int(label) == 1:
        return "Seizure"
    return "Non-seizure"


def build_prompt_messages(image):
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_PROMPT}],
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": USER_PROMPT},
                {"type": "image", "image": image},
            ],
        },
    ]


def build_training_messages(image, target_text):
    messages = build_prompt_messages(image)
    messages.append(
        {
            "role": "assistant",
            "content": [{"type": "text", "text": target_text}],
        }
    )
    return messages


class MedGemmaTrainingCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        if len(examples) != 1:
            raise ValueError("This pipeline expects physical batch size = 1.")

        example = examples[0]
        image_path = example["image_path"]
        target_text = label_to_target_text(example["label"])

        with Image.open(image_path) as opened_image:
            image = opened_image.convert("RGB")

            prompt_messages = build_prompt_messages(image)
            prompt_inputs = self.processor.apply_chat_template(
                prompt_messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )

            full_messages = build_training_messages(image, target_text)
            full_inputs = self.processor.apply_chat_template(
                full_messages,
                add_generation_prompt=False,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )

        prompt_ids = prompt_inputs["input_ids"]
        full_ids = full_inputs["input_ids"]
        prompt_length = int(prompt_ids.shape[1])

        if full_ids.shape[1] <= prompt_length:
            raise RuntimeError("Full training conversation contains no target response.")

        prefix_matches = torch.equal(full_ids[:, :prompt_length], prompt_ids)
        if not prefix_matches:
            raise RuntimeError("MedGemma chat template prefix mismatch. Training stopped.")

        labels = full_ids.clone()
        labels[:, :prompt_length] = -100

        if int((labels != -100).sum()) == 0:
            raise RuntimeError("No assistant target tokens remain after masking.")

        full_inputs["labels"] = labels
        return full_inputs


train_dataset = EEGTopomapDataset(actual_train_df)
validation_dataset = EEGTopomapDataset(validation_df)
training_collator = MedGemmaTrainingCollator(processor)

train_generator = torch.Generator()
train_generator.manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=NUM_WORKERS,
    collate_fn=training_collator,
)

validation_loss_loader = DataLoader(
    validation_dataset,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=training_collator,
)


def move_batch_to_device(batch):
    moved = {}
    for key, value in batch.items():
        if not torch.is_tensor(value):
            moved[key] = value
        elif value.dtype.is_floating_point:
            moved[key] = value.to(device=DEVICE, dtype=MODEL_DTYPE)
        else:
            moved[key] = value.to(DEVICE)
    return moved


def autocast_context():
    return torch.autocast(device_type="cuda", dtype=MODEL_DTYPE)


# %%
# ==================================================================================================
# 11. DIAGNOSTICS, INFERENCE, METRICS
# ==================================================================================================

def arm_all_precision_diagnostics(interventions):
    for intervention in interventions.values():
        intervention.arm_precision_diagnostic()


def print_all_precision_diagnostics(interventions, title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    for layer_key in sorted(interventions.keys(), key=lambda x: int(x)):
        stats = interventions[layer_key].get_precision_diagnostic()
        if stats is None:
            print(f"Layer {layer_key}: no diagnostic sample was captured.")
            continue

        print(f"Layer {layer_key}:")
        print(f"  Base hidden dtype             : {stats['base_dtype']}")
        print(f"  LoReFT computation dtype      : {stats['reft_dtype']}")
        print(f"  Raw FP32 delta L2 norm        : {stats['raw_fp32_delta_norm']:.12e}")
        print(f"  Effective post-cast delta norm: {stats['effective_post_cast_delta_norm']:.12e}")
        print(f"  BF16 cast error L2 norm       : {stats['cast_error_norm']:.12e}")
        print(f"  Retained delta-norm ratio     : {stats['retained_norm_ratio']:.6f}")
        print(f"  Hidden elements changed       : {100.0 * stats['changed_element_fraction']:.4f}%")
        print(f"  Raw FP32 mean |delta|         : {stats['raw_fp32_delta_mean_abs']:.12e}")
        print(f"  Effective mean |delta|        : {stats['effective_post_cast_delta_mean_abs']:.12e}")

    print("=" * 100)


def normalize_prediction_text(output_text):
    text = str(output_text).strip().lower()
    text = text.replace("_", " ").replace("–", "-")
    text = re.sub(r"\s+", " ", text)

    if re.search(r"\bnon[\s-]*seizure\b", text):
        return 0
    if re.search(r"\bseizure\b", text):
        return 1
    return None


@torch.no_grad()
def predict_single_image(image_path):
    model.eval()

    with Image.open(image_path) as opened_image:
        image = opened_image.convert("RGB")
        messages = build_prompt_messages(image)
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

    input_length = int(inputs["input_ids"].shape[1])
    inputs = move_batch_to_device(inputs)

    with autocast_context():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    generated_tokens = generated[0, input_length:]
    decoded = processor.decode(generated_tokens, skip_special_tokens=True).strip()
    prediction = normalize_prediction_text(decoded)
    return decoded, prediction


def calculate_binary_metrics(true_labels, predictions):
    metric_predictions = []
    invalid_count = 0

    for truth, prediction in zip(true_labels, predictions):
        if prediction is None:
            invalid_count += 1
            metric_predictions.append(1 - int(truth))
        else:
            metric_predictions.append(int(prediction))

    y_true = np.asarray(true_labels, dtype=int)
    y_pred = np.asarray(metric_predictions, dtype=int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metrics = {
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Sensitivity": float(sensitivity),
        "Specificity": float(specificity),
        "Balanced_Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Invalid_Predictions": int(invalid_count),
    }

    return metrics, y_pred


@torch.no_grad()
def calculate_validation_loss():
    model.eval()
    losses = []

    progress = tqdm(validation_loss_loader, desc="Validation loss", leave=False)
    for batch in progress:
        batch = move_batch_to_device(batch)

        with autocast_context():
            outputs = model(**batch)

        loss_value = float(outputs.loss.detach().float().cpu())
        losses.append(loss_value)
        progress.set_postfix(loss=f"{loss_value:.4f}")

        del batch
        del outputs

    return float(np.mean(losses))


@torch.no_grad()
def evaluate_validation_generation(epoch):
    model.eval()
    true_labels = []
    predictions = []
    rows = []

    progress = tqdm(range(len(validation_df)), desc=f"Validation predictions epoch {epoch}")
    for index in progress:
        row = validation_df.iloc[index]
        image_path = str(row["_image_path"])

        raw_output, prediction = predict_single_image(image_path)
        true_label = int(row["_label"])

        true_labels.append(true_label)
        predictions.append(prediction)

        rows.append(
            {
                "index": index,
                "image_path": image_path,
                "true_label": true_label,
                "true_class": "Seizure" if true_label == 1 else "Non-seizure",
                "raw_model_output": raw_output,
                "prediction": prediction if prediction is not None else "INVALID",
                "predicted_class": (
                    "Seizure"
                    if prediction == 1
                    else "Non-seizure"
                    if prediction == 0
                    else "INVALID"
                ),
            }
        )

    metrics, metric_predictions = calculate_binary_metrics(true_labels, predictions)
    validation_prediction_df = pd.DataFrame(rows)
    validation_prediction_df["metric_prediction"] = metric_predictions
    validation_prediction_df["correct"] = validation_prediction_df["true_label"].to_numpy() == metric_predictions

    validation_output_csv = os.path.join(OUTPUT_ROOT, f"validation_predictions_epoch_{epoch}.csv")
    validation_prediction_df.to_csv(validation_output_csv, index=False)

    return metrics


# %%
# ==================================================================================================
# 12. PRE-FLIGHT TEST
# ==================================================================================================

print("\n" + "=" * 100)
print("LoReFT R4 PRE-FLIGHT TEST")
print("=" * 100)

preview_example = train_dataset[0]
print("\nImage:", preview_example["image_path"])
print("Target:", label_to_target_text(preview_example["label"]))

preview_batch_cpu = training_collator([preview_example])
target_ids = preview_batch_cpu["labels"][0]
target_ids = target_ids[target_ids != -100]

print("Assistant target tokens:", len(target_ids))

try:
    print("Decoded supervised target:", repr(processor.decode(target_ids, skip_special_tokens=False)))
except Exception:
    pass

preview_batch_gpu = move_batch_to_device(preview_batch_cpu)
model.eval()
arm_all_precision_diagnostics(loreft_interventions)

with torch.no_grad():
    with autocast_context():
        preview_output = model(**preview_batch_gpu)

preview_loss = float(preview_output.loss.detach().float().cpu())
print(f"Pre-flight loss: {preview_loss:.6f}")

print_all_precision_diagnostics(
    loreft_interventions,
    "PRE-FLIGHT TRUE-FP32 LoReFT INTERVENTION DIAGNOSTIC",
)

if not np.isfinite(preview_loss):
    raise RuntimeError("Pre-flight loss is not finite.")

print("\nSUCCESS: LoReFT R4 forward pass works.")

del preview_batch_cpu
del preview_batch_gpu
del preview_output
gc.collect()
torch.cuda.empty_cache()


# %%
# ==================================================================================================
# 13. OPTIMIZER, SCHEDULER, UPDATE DIAGNOSTICS
# ==================================================================================================

trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]

rotation_parameters = []
source_weight_parameters = []
source_bias_parameters = []

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    if "rotate_layer.parametrizations.weight.original" in name:
        rotation_parameters.append(parameter)
    elif name.endswith("learned_source.bias"):
        source_bias_parameters.append(parameter)
    elif "learned_source.weight" in name:
        source_weight_parameters.append(parameter)
    else:
        raise RuntimeError(f"Unexpected trainable LoReFT parameter in optimizer grouping: {name}")

if len(rotation_parameters) != NUM_LOREFT_INTERVENTIONS:
    raise RuntimeError(
        f"Expected one trainable orthogonal-rotation parameter per LoReFT layer. "
        f"Expected {NUM_LOREFT_INTERVENTIONS}, found {len(rotation_parameters)}."
    )

optimizer = torch.optim.AdamW(
    [
        {"params": rotation_parameters, "weight_decay": 0.0, "group_name": "loreft_rotation_no_decay"},
        {"params": source_weight_parameters, "weight_decay": WEIGHT_DECAY, "group_name": "loreft_source_weight"},
        {"params": source_bias_parameters, "weight_decay": 0.0, "group_name": "loreft_source_bias_no_decay"},
    ],
    lr=LEARNING_RATE,
)

print("\nLoReFT optimizer parameter groups:")
for group in optimizer.param_groups:
    print(
        f"  {group.get('group_name', 'unnamed')}: "
        f"weight_decay={group['weight_decay']}, "
        f"parameters={sum(p.numel() for p in group['params']):,}"
    )


def snapshot_trainable_parameters():
    return [parameter.detach().float().cpu().clone() for parameter in trainable_parameters]


def effective_rotation_norms():
    norms = {}
    for layer_key, intervention in loreft_interventions.items():
        with torch.no_grad():
            effective_rotation = intervention.rotate_layer.weight.detach().float()
            norms[layer_key] = float(effective_rotation.norm().cpu())
    return norms


def print_effective_rotation_norms(title):
    norms = effective_rotation_norms()
    expected = float(LOREFT_RANK ** 0.5)

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    for layer_key in sorted(norms.keys(), key=lambda x: int(x)):
        value = norms[layer_key]
        print(f"Layer {layer_key}: ||R||_F = {value:.12e} (expected ~= {expected:.6f})")

        if not np.isfinite(value) or value < 0.5 * expected:
            raise RuntimeError(
                f"Effective LoReFT rotation collapsed at layer {layer_key}. "
                f"Observed ||R||_F={value:.12e}, expected near {expected:.6f}."
            )

    print("=" * 100)


def trainable_gradient_l2_norm():
    total_sq = 0.0
    nonzero_tensors = 0
    tensors_with_grad = 0

    for parameter in trainable_parameters:
        if parameter.grad is None:
            continue

        tensors_with_grad += 1
        grad = parameter.grad.detach().float()
        total_sq += float(torch.sum(grad * grad).cpu())

        if bool(torch.any(grad != 0).item()):
            nonzero_tensors += 1

    return math.sqrt(total_sq), tensors_with_grad, nonzero_tensors


def parameter_change_from_snapshot(before_snapshot):
    total_sq = 0.0
    max_abs = 0.0

    for before, parameter in zip(before_snapshot, trainable_parameters):
        after = parameter.detach().float().cpu()
        delta = after - before
        total_sq += float(torch.sum(delta * delta))
        if delta.numel() > 0:
            max_abs = max(max_abs, float(delta.abs().max()))

    return math.sqrt(total_sq), max_abs


optimizer_steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
total_optimizer_steps = optimizer_steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_optimizer_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps,
)

if MODEL_DTYPE == torch.float16:
    try:
        grad_scaler = torch.amp.GradScaler("cuda")
    except Exception:
        grad_scaler = torch.cuda.amp.GradScaler()
else:
    grad_scaler = None

print("\n" + "=" * 100)
print("LoReFT R4 TRAINING SETTINGS")
print("=" * 100)
print(f"\nEpochs                    : {NUM_EPOCHS}")
print(f"Training images           : {len(train_dataset)}")
print(f"Validation images         : {len(validation_dataset)}")
print(f"Physical batch size       : {TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation     : {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size      : {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate             : {LEARNING_RATE}")
print(f"Warmup ratio              : {WARMUP_RATIO}")
print(f"Warmup optimizer steps    : {warmup_steps}")
print(f"LoReFT rank               : {LOREFT_RANK}")
print(f"Language layers           : {NUM_LANGUAGE_LAYERS}")
print(f"Intervention human layers : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Intervention Python idx   : {LOREFT_TARGET_LAYER_INDICES}")
print(f"Prefix prompt tokens      : {LOREFT_PREFIX_TOKENS}")
print(f"Suffix prompt tokens      : {LOREFT_SUFFIX_TOKENS}")
print(f"LoReFT dropout            : {LOREFT_DROPOUT}")
print("Base MedGemma weights     : FROZEN")
print("Vision tower              : FROZEN")
print("Gradient checkpointing    : OFF")
print("LoReFT parameter dtype    : float32")
print("LoReFT compute dtype      : float32")
print("Rotation weight decay     : 0.0")
print(f"Source weight decay       : {WEIGHT_DECAY}")
print(f"Run ID                    : {RUN_ID}")
print(f"Output folder             : {OUTPUT_ROOT}")

print_effective_rotation_norms("PRE-TRAIN EFFECTIVE LoReFT ROTATION-NORM CHECK")


# %%
# ==================================================================================================
# 14. SAVE EXPERIMENT CONFIGURATION
# ==================================================================================================

experiment_config = {
    "experiment": "Pooled_A2D2_R4",
    "patients_in_training_pool": ["08", "11", "18", "24"],
    "representation": "A2D2-coefficient fixed-scale EEG variance topomap",
    "model": MODEL_ID,
    "fine_tuning_method": "LoReFT",
    "loreft_formula": "h + R^T(Wh + b - Rh)",
    "quantization": "None",
    "random_seed": RANDOM_SEED,
    "training_metadata_csv": TRAINING_METADATA_CSV,
    "test_metadata_csvs": TEST_METADATA_CSVS,
    "system_prompt": SYSTEM_PROMPT,
    "user_prompt": USER_PROMPT,
    "training_pool_images": len(full_training_df),
    "actual_training_images": len(actual_train_df),
    "validation_images": len(validation_df),
    "loreft_rank": LOREFT_RANK,
    "num_language_layers": NUM_LANGUAGE_LAYERS,
    "loreft_target_layers_human": LOREFT_TARGET_LAYERS_HUMAN,
    "loreft_target_layer_python_indices": LOREFT_TARGET_LAYER_INDICES,
    "loreft_num_interventions": len(LOREFT_TARGET_LAYER_INDICES),
    "language_layers_path": LANGUAGE_LAYERS_PATH,
    "hidden_size": LOREFT_HIDDEN_SIZE,
    "prefix_prompt_tokens": LOREFT_PREFIX_TOKENS,
    "suffix_prompt_tokens": LOREFT_SUFFIX_TOKENS,
    "loreft_dropout": LOREFT_DROPOUT,
    "component": LOREFT_COMPONENT,
    "gradient_checkpointing": False,
    "loreft_train_dtype": "float32",
    "checkpoint_save_strategy": "new directory per best epoch; no rmtree",
    "epochs": NUM_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "best_model_primary_criterion": "Highest validation balanced accuracy",
    "best_model_tie_breaker": "Lower validation loss",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "physical_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "trainable_parameters": trainable_parameter_count,
    "total_parameters": total_parameter_count,
    "trainable_percentage": trainable_percentage,
    "precision": PRECISION_NAME,
    "gpu": GPU_NAME,
    "gpu_memory_gb": GPU_MEMORY_GB,
    "pytorch_version": torch.__version__,
    "transformers_version": transformers.__version__,
}

with open(EXPERIMENT_CONFIG_JSON, "w", encoding="utf-8") as file:
    json.dump(experiment_config, file, indent=4)


# %%
# ==================================================================================================
# 15. FORCE EPOCH 11 AS THE SELECTED BEST CHECKPOINT + VALIDATION
# ==================================================================================================

# This script intentionally skips training. It loads the saved LoReFT checkpoint from epoch 11,
# treats it as the selected best checkpoint, evaluates the pooled validation split, and then runs
# the patient-wise blinded tests using a fresh MedGemma model with the same epoch-11 LoReFT weights.

FORCED_BEST_EPOCH = 11
FORCED_CHECKPOINT_RUN_ROOT = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_023326"
)

BEST_ADAPTER_DIR = os.path.join(FORCED_CHECKPOINT_RUN_ROOT, "ckpt", f"e{FORCED_BEST_EPOCH:03d}")
BEST_REFT_WEIGHTS = os.path.join(BEST_ADAPTER_DIR, "loreft_intervention.pt")
BEST_REFT_CONFIG = os.path.join(BEST_ADAPTER_DIR, "loreft_config.json")

FORCED_VALIDATION_METRICS_JSON = os.path.join(
    OUTPUT_ROOT,
    f"forced_epoch_{FORCED_BEST_EPOCH:03d}_validation_metrics.json",
)
FORCED_VALIDATION_METRICS_CSV = os.path.join(
    OUTPUT_ROOT,
    f"forced_epoch_{FORCED_BEST_EPOCH:03d}_validation_metrics.csv",
)
FORCED_VALIDATION_CONFUSION_CSV = os.path.join(
    OUTPUT_ROOT,
    f"forced_epoch_{FORCED_BEST_EPOCH:03d}_validation_confusion_matrix.csv",
)

if not os.path.isfile(BEST_REFT_WEIGHTS):
    raise FileNotFoundError(f"Epoch-11 LoReFT weights were not found:\n{BEST_REFT_WEIGHTS}")

if not os.path.isfile(BEST_REFT_CONFIG):
    raise FileNotFoundError(f"Epoch-11 LoReFT config was not found:\n{BEST_REFT_CONFIG}")

print("\n" + "=" * 100)
print("FORCING EPOCH 11 AS THE SELECTED BEST LoReFT CHECKPOINT")
print("=" * 100)
print(f"\nForced best epoch   : {FORCED_BEST_EPOCH}")
print(f"Checkpoint directory: {BEST_ADAPTER_DIR}")
print(f"Weights             : {BEST_REFT_WEIGHTS}")
print(f"Config              : {BEST_REFT_CONFIG}")

with open(BEST_REFT_CONFIG, "r", encoding="utf-8") as file:
    best_reft_config = json.load(file)

try:
    saved_reft_state = torch.load(BEST_REFT_WEIGHTS, map_location=DEVICE, weights_only=True)
except TypeError:
    saved_reft_state = torch.load(BEST_REFT_WEIGHTS, map_location=DEVICE)

if not isinstance(saved_reft_state, dict):
    raise RuntimeError("Saved multi-layer LoReFT state is not a dictionary.")

for layer_key, intervention in loreft_interventions.items():
    if layer_key not in saved_reft_state:
        raise RuntimeError(f"Missing LoReFT state for human layer {layer_key} in epoch-11 checkpoint.")
    intervention.load_state_dict(saved_reft_state[layer_key], strict=True)
    intervention.to(DEVICE)

best_epoch = FORCED_BEST_EPOCH
stopped_early = False
early_stop_epoch = None
training_history = []
total_training_minutes = 0.0
peak_training_memory_gb = torch.cuda.max_memory_allocated(DEVICE) / 1024**3

print("\nEpoch-11 LoReFT weights loaded into the current model.")
print_effective_rotation_norms("FORCED EPOCH 11 EFFECTIVE LoReFT ROTATION-NORM CHECK")

print("\nCalculating pooled validation loss for forced epoch 11...")
arm_all_precision_diagnostics(loreft_interventions)
best_validation_loss = calculate_validation_loss()
print_all_precision_diagnostics(
    loreft_interventions,
    "FORCED EPOCH 11 FIRST-VALIDATION-BATCH TRUE-FP32 LoReFT DIAGNOSTIC",
)

print("\nRunning pooled validation generation/classification for forced epoch 11...")
validation_metrics = evaluate_validation_generation(best_epoch)
best_validation_balanced_accuracy = validation_metrics["Balanced_Accuracy"]

validation_confusion_df = pd.DataFrame(
    [
        [validation_metrics["TN"], validation_metrics["FP"]],
        [validation_metrics["FN"], validation_metrics["TP"]],
    ],
    index=["Actual_Non_Seizure", "Actual_Seizure"],
    columns=["Predicted_Non_Seizure", "Predicted_Seizure"],
)
validation_confusion_df.to_csv(FORCED_VALIDATION_CONFUSION_CSV)

forced_validation_summary = {
    "Evaluation_Mode": "Forced epoch-11 checkpoint; training skipped",
    "Experiment": "Pooled_A2D2_R4_Epoch11_Selected",
    "Forced_Best_Epoch": best_epoch,
    "Best_Checkpoint_Directory": BEST_ADAPTER_DIR,
    "Best_ReFT_Weights": BEST_REFT_WEIGHTS,
    "Best_ReFT_Config": BEST_REFT_CONFIG,
    "Validation_Loss": best_validation_loss,
    "TN": validation_metrics["TN"],
    "FP": validation_metrics["FP"],
    "FN": validation_metrics["FN"],
    "TP": validation_metrics["TP"],
    "Sensitivity": validation_metrics["Sensitivity"],
    "Specificity": validation_metrics["Specificity"],
    "Balanced_Accuracy": validation_metrics["Balanced_Accuracy"],
    "Accuracy": validation_metrics["Accuracy"],
    "Precision": validation_metrics["Precision"],
    "F1": validation_metrics["F1"],
    "Invalid_Predictions": validation_metrics["Invalid_Predictions"],
    "Validation_Confusion_CSV": FORCED_VALIDATION_CONFUSION_CSV,
}

with open(FORCED_VALIDATION_METRICS_JSON, "w", encoding="utf-8") as file:
    json.dump(forced_validation_summary, file, indent=4)

pd.DataFrame([forced_validation_summary]).to_csv(FORCED_VALIDATION_METRICS_CSV, index=False)
training_history.append(
    {
        "epoch": best_epoch,
        "training_loss": None,
        "validation_loss": best_validation_loss,
        "TN": validation_metrics["TN"],
        "FP": validation_metrics["FP"],
        "FN": validation_metrics["FN"],
        "TP": validation_metrics["TP"],
        "sensitivity": validation_metrics["Sensitivity"],
        "specificity": validation_metrics["Specificity"],
        "balanced_accuracy": validation_metrics["Balanced_Accuracy"],
        "accuracy": validation_metrics["Accuracy"],
        "precision": validation_metrics["Precision"],
        "f1": validation_metrics["F1"],
        "invalid_predictions": validation_metrics["Invalid_Predictions"],
        "epoch_minutes": None,
        "evaluation_mode": "forced_epoch_11_training_skipped",
    }
)
pd.DataFrame(training_history).to_csv(TRAINING_HISTORY_CSV, index=False)

experiment_config.update(
    {
        "evaluation_mode": "Forced epoch-11 checkpoint; training skipped",
        "forced_best_epoch": best_epoch,
        "forced_checkpoint_run_root": FORCED_CHECKPOINT_RUN_ROOT,
        "best_checkpoint_directory": BEST_ADAPTER_DIR,
        "best_reft_weights": BEST_REFT_WEIGHTS,
        "best_reft_config": BEST_REFT_CONFIG,
        "best_validation_balanced_accuracy": best_validation_balanced_accuracy,
        "best_validation_loss": best_validation_loss,
        "forced_validation_metrics_json": FORCED_VALIDATION_METRICS_JSON,
        "forced_validation_metrics_csv": FORCED_VALIDATION_METRICS_CSV,
        "forced_validation_confusion_csv": FORCED_VALIDATION_CONFUSION_CSV,
    }
)

with open(EXPERIMENT_CONFIG_JSON, "w", encoding="utf-8") as file:
    json.dump(experiment_config, file, indent=4)

print("\n" + "=" * 100)
print("FORCED EPOCH 11 POOLED VALIDATION RESULTS")
print("=" * 100)
print(f"Validation images   : {len(validation_df)}")
print(f"Validation loss     : {best_validation_loss:.6f}")
print(f"TN / FP / FN / TP   : {validation_metrics['TN']} / {validation_metrics['FP']} / {validation_metrics['FN']} / {validation_metrics['TP']}")
print(f"Sensitivity         : {validation_metrics['Sensitivity']:.4f}")
print(f"Specificity         : {validation_metrics['Specificity']:.4f}")
print(f"Balanced Accuracy   : {validation_metrics['Balanced_Accuracy']:.4f}")
print(f"Accuracy            : {validation_metrics['Accuracy']:.4f}")
print(f"Precision           : {validation_metrics['Precision']:.4f}")
print(f"F1                  : {validation_metrics['F1']:.4f}")
print(f"Invalid outputs     : {validation_metrics['Invalid_Predictions']}")
print("\nValidation confusion matrix:")
print(validation_confusion_df.to_string())
print("\nSaved forced validation metrics:")
print(FORCED_VALIDATION_METRICS_JSON)
print(FORCED_VALIDATION_METRICS_CSV)
print(FORCED_VALIDATION_CONFUSION_CSV)


# %%
# ==================================================================================================
# 16. RELEASE CURRENT MODEL BEFORE PATIENT-WISE EVALUATION
# ==================================================================================================

print("\n" + "=" * 100)
print("FORCED EPOCH 11 VALIDATION COMPLETE; RELEASING CURRENT MODEL")
print("=" * 100)

for hook_handle in loreft_hook_handles:
    try:
        hook_handle.remove()
    except Exception:
        pass

del model
del base_model
del optimizer
del scheduler
del trainable_parameters
del loreft_info
del target_language_layers
del loreft_hook_handles
del loreft_interventions
del loreft_controller

gc.collect()
torch.cuda.empty_cache()


# %%
# ==================================================================================================
# 17. LOAD FRESH MEDGEMMA + FORCED EPOCH 11 LoReFT CHECKPOINT
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING FRESH MEDGEMMA FOR PATIENT-WISE EPOCH 11 EVALUATION")
print("=" * 100)

base_model_for_test = load_medgemma_base()

if hasattr(base_model_for_test.config, "use_cache"):
    base_model_for_test.config.use_cache = True

for parameter in base_model_for_test.parameters():
    parameter.requires_grad = False

with open(BEST_REFT_CONFIG, "r", encoding="utf-8") as file:
    best_reft_config = json.load(file)

try:
    saved_reft_state = torch.load(BEST_REFT_WEIGHTS, map_location=DEVICE, weights_only=True)
except TypeError:
    saved_reft_state = torch.load(BEST_REFT_WEIGHTS, map_location=DEVICE)

if not isinstance(saved_reft_state, dict):
    raise RuntimeError("Saved multi-layer LoReFT state is not a dictionary.")

model, test_loreft_info, test_loreft_controller, test_loreft_interventions, test_loreft_hook_handles, test_target_layers, _ = attach_loreft(
    base=base_model_for_test,
    target_layers_human=[int(x) for x in best_reft_config["target_layers_human"]],
    rank=int(best_reft_config["rank"]),
    prefix_tokens=int(best_reft_config["prefix_prompt_tokens"]),
    suffix_tokens=int(best_reft_config["suffix_prompt_tokens"]),
    dropout=float(best_reft_config["dropout"]),
    trainable=False,
    saved_state=saved_reft_state,
    saved_config=best_reft_config,
)

model.eval()

if test_loreft_info["num_language_layers"] != int(best_reft_config["num_language_layers"]):
    raise RuntimeError("Language-layer count differs between the epoch-11 checkpoint and final test model.")

print(f"\nForced epoch-11 LoReFT R4 checkpoint loaded for patient-wise evaluation.")
print(
    f"Target layers: human {best_reft_config['target_layers_human']} / "
    f"{best_reft_config['num_language_layers']}"
)


# %%
# ==================================================================================================
# 18. FINAL BLINDED TEST LOOP
# ==================================================================================================

@torch.no_grad()
def evaluate_blinded_patient(patient_id, testing_df):
    print("\n" + "=" * 100)
    print(f"STARTING PATIENT {patient_id} A2D2 BLINDED TEST")
    print("=" * 100)

    patient_output_dir = os.path.join(TEST_OUTPUT_ROOT, f"patient_{patient_id}")
    os.makedirs(patient_output_dir, exist_ok=True)

    predictions_csv = os.path.join(patient_output_dir, f"patient_{patient_id}_predictions.csv")
    metrics_json = os.path.join(patient_output_dir, f"patient_{patient_id}_metrics.json")
    confusion_csv = os.path.join(patient_output_dir, f"patient_{patient_id}_confusion.csv")

    rows = []
    true_labels = []
    predictions = []
    test_start_time = time.time()

    progress = tqdm(range(len(testing_df)), desc=f"Patient {patient_id} blinded test")
    for index in progress:
        row = testing_df.iloc[index]
        image_path = str(row["_image_path"])

        raw_output, prediction = predict_single_image(image_path)
        true_label = int(row["_label"])

        true_labels.append(true_label)
        predictions.append(prediction)

        rows.append(
            {
                "patient_id": patient_id,
                "test_index": index,
                "image_path": image_path,
                "true_label": true_label,
                "true_class": "Seizure" if true_label == 1 else "Non-seizure",
                "raw_model_output": raw_output,
                "normalized_prediction": prediction if prediction is not None else "INVALID",
                "predicted_class": (
                    "Seizure"
                    if prediction == 1
                    else "Non-seizure"
                    if prediction == 0
                    else "INVALID"
                ),
            }
        )

        if (index + 1) % 25 == 0 or (index + 1) == len(testing_df):
            pd.DataFrame(rows).to_csv(predictions_csv, index=False)

    final_metrics, metric_predictions = calculate_binary_metrics(true_labels, predictions)

    prediction_df = pd.DataFrame(rows)
    prediction_df["metric_prediction"] = metric_predictions
    prediction_df["correct"] = prediction_df["true_label"].to_numpy() == metric_predictions

    original_metadata = testing_df.drop(columns=["_image_path", "_label"], errors="ignore").add_prefix("metadata_")
    prediction_df = pd.concat(
        [
            prediction_df.reset_index(drop=True),
            original_metadata.reset_index(drop=True),
        ],
        axis=1,
    )
    prediction_df.to_csv(predictions_csv, index=False)

    cm_df = pd.DataFrame(
        [
            [final_metrics["TN"], final_metrics["FP"]],
            [final_metrics["FN"], final_metrics["TP"]],
        ],
        index=["Actual_Non_Seizure", "Actual_Seizure"],
        columns=["Predicted_Non_Seizure", "Predicted_Seizure"],
    )
    cm_df.to_csv(confusion_csv)

    final_test_minutes = (time.time() - test_start_time) / 60.0

    final_metrics.update(
        {
            "Experiment": "Pooled_A2D2_R4",
            "Patient": patient_id,
            "Base_Model": MODEL_ID,
            "Representation": "A2D2-coefficient fixed-scale EEG variance topomap",
            "Method": "LoReFT",
            "Quantization": "None",
            "LoReFT_Rank": LOREFT_RANK,
            "LoReFT_Target_Layers_Human": LOREFT_TARGET_LAYERS_HUMAN,
            "LoReFT_Target_Layer_Python_Indices": LOREFT_TARGET_LAYER_INDICES,
            "LoReFT_Number_Interventions": len(LOREFT_TARGET_LAYER_INDICES),
            "LoReFT_Prefix_Prompt_Tokens": LOREFT_PREFIX_TOKENS,
            "LoReFT_Suffix_Prompt_Tokens": LOREFT_SUFFIX_TOKENS,
            "LoReFT_Dropout": LOREFT_DROPOUT,
            "Gradient_Checkpointing": False,
            "LoReFT_Train_Dtype": "float32",
            "Best_Checkpoint_Directory": BEST_ADAPTER_DIR,
            "Best_Epoch": best_epoch,
            "Best_Validation_Balanced_Accuracy": best_validation_balanced_accuracy,
            "Best_Validation_Loss": best_validation_loss,
            "Trainable_Parameters": trainable_parameter_count,
            "Total_Parameters": total_parameter_count,
            "Trainable_Percentage": trainable_percentage,
            "Training_Time_Minutes": total_training_minutes,
            "Final_Test_Time_Minutes": final_test_minutes,
            "Peak_Training_GPU_GB": peak_training_memory_gb,
            "Test_Metadata_CSV": TEST_METADATA_CSVS[patient_id],
            "Test_Total": len(testing_df),
            "Test_Seizure": int((testing_df["_label"] == 1).sum()),
            "Test_Non_Seizure": int((testing_df["_label"] == 0).sum()),
            "Predictions_CSV": predictions_csv,
            "Metrics_JSON": metrics_json,
            "Confusion_CSV": confusion_csv,
        }
    )

    with open(metrics_json, "w", encoding="utf-8") as file:
        json.dump(final_metrics, file, indent=4)

    print("\n" + "-" * 100)
    print(f"PATIENT {patient_id} FINAL RESULTS")
    print("-" * 100)
    print(f"Test images         : {len(testing_df)}")
    print(f"TN / FP / FN / TP   : {final_metrics['TN']} / {final_metrics['FP']} / {final_metrics['FN']} / {final_metrics['TP']}")
    print(f"Sensitivity         : {final_metrics['Sensitivity']:.4f}")
    print(f"Specificity         : {final_metrics['Specificity']:.4f}")
    print(f"Balanced Accuracy   : {final_metrics['Balanced_Accuracy']:.4f}")
    print(f"Accuracy            : {final_metrics['Accuracy']:.4f}")
    print(f"Precision           : {final_metrics['Precision']:.4f}")
    print(f"F1                  : {final_metrics['F1']:.4f}")
    print(f"Invalid outputs     : {final_metrics['Invalid_Predictions']}")
    print(f"Predictions CSV     : {predictions_csv}")
    print(f"Metrics JSON        : {metrics_json}")
    print(f"Confusion CSV       : {confusion_csv}")

    return final_metrics


all_patient_metrics = []

for patient_id in ["08", "11", "18", "24"]:
    patient_metrics = evaluate_blinded_patient(patient_id, testing_dfs[patient_id])
    all_patient_metrics.append(patient_metrics)
    gc.collect()
    torch.cuda.empty_cache()

all_metrics_df = pd.DataFrame(all_patient_metrics)
all_metrics_df.to_csv(ALL_PATIENT_METRICS_CSV, index=False)

with open(ALL_PATIENT_METRICS_JSON, "w", encoding="utf-8") as file:
    json.dump(all_patient_metrics, file, indent=4)


# %%
# ==================================================================================================
# 19. UPDATE CONFIG + FINAL REPORT
# ==================================================================================================

experiment_config["best_epoch"] = best_epoch
experiment_config["best_validation_balanced_accuracy"] = best_validation_balanced_accuracy
experiment_config["best_validation_loss"] = best_validation_loss
experiment_config["training_time_minutes"] = total_training_minutes
experiment_config["actual_epochs_run"] = len(training_history)
experiment_config["stopped_early"] = stopped_early
experiment_config["early_stop_epoch"] = early_stop_epoch
experiment_config["final_blinded_test_patients"] = ["08", "11", "18", "24"]
experiment_config["all_patient_metrics_csv"] = ALL_PATIENT_METRICS_CSV
experiment_config["all_patient_metrics_json"] = ALL_PATIENT_METRICS_JSON

with open(EXPERIMENT_CONFIG_JSON, "w", encoding="utf-8") as file:
    json.dump(experiment_config, file, indent=4)

print("\n" + "=" * 100)
print("POOLED A2D2 MEDGEMMA-4B LoReFT R4 COMPLETE")
print("=" * 100)

print("\nLoReFT R4 CONFIGURATION")
print(f"Rank                 : {LOREFT_RANK}")
print(f"Language layers      : {NUM_LANGUAGE_LAYERS}")
print(f"Target human layers  : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Target Python indices: {LOREFT_TARGET_LAYER_INDICES}")
print(f"Interventions        : {len(LOREFT_TARGET_LAYER_INDICES)}")
print(f"Prefix prompt tokens : {LOREFT_PREFIX_TOKENS}")
print(f"Suffix prompt tokens : {LOREFT_SUFFIX_TOKENS}")
print(f"Dropout              : {LOREFT_DROPOUT}")
print("Gradient checkpoint  : OFF")
print("LoReFT parameter dtype: float32")
print("LoReFT compute dtype : float32")
print("Initialization       : identity/no-op")
print("Rotation weight decay: 0.0")
print(f"Source weight decay  : {WEIGHT_DECAY}")
print(f"Learning rate        : {LEARNING_RATE}")
print(f"Epochs               : {NUM_EPOCHS}")
print(f"Early stop patience  : {EARLY_STOPPING_PATIENCE}")
print(f"Actual epochs run    : {len(training_history)}")
print(f"Stopped early        : {stopped_early}")

print("\nDATA")
print(
    f"Training             : {int((actual_train_df['_label'] == 1).sum())} seizure + "
    f"{int((actual_train_df['_label'] == 0).sum())} non-seizure = {len(actual_train_df)}"
)
print(
    f"Validation           : {int((validation_df['_label'] == 1).sum())} seizure + "
    f"{int((validation_df['_label'] == 0).sum())} non-seizure = {len(validation_df)}"
)

print("\nBEST CHECKPOINT")
print(f"Best epoch           : {best_epoch}")
print(f"Best validation BA   : {best_validation_balanced_accuracy:.4f}")
print(f"Best validation loss : {best_validation_loss:.6f}")
print(f"Best checkpoint      : {BEST_ADAPTER_DIR}")

print("\nPATIENT-WISE FINAL TEST SUMMARY")
summary_columns = [
    "Patient",
    "Test_Total",
    "TN",
    "FP",
    "FN",
    "TP",
    "Sensitivity",
    "Specificity",
    "Balanced_Accuracy",
    "Accuracy",
    "Precision",
    "F1",
    "Invalid_Predictions",
]
print(all_metrics_df[summary_columns].to_string(index=False))

print("\nFILES SAVED")
print("Training split       :", TRAIN_SPLIT_CSV)
print("Validation split     :", VALIDATION_SPLIT_CSV)
print("Training history     :", TRAINING_HISTORY_CSV)
print("Experiment config    :", EXPERIMENT_CONFIG_JSON)
print("Target info          :", TARGET_MODULES_TXT)
print("All metrics CSV      :", ALL_PATIENT_METRICS_CSV)
print("All metrics JSON     :", ALL_PATIENT_METRICS_JSON)
print("Per-patient outputs  :", TEST_OUTPUT_ROOT)
print("\n" + "=" * 100)


W1011 15:42:57.647000 25196 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels



HUGGING FACE LOGIN CHECK

Logged in as: sd377

GPU CHECK

GPU: NVIDIA GeForce RTX 5080
GPU memory: 15.92 GB
Model/training dtype: bfloat16

PACKAGE VERSIONS
Python       : 3.11.15
PyTorch      : 2.11.0+cu128
Transformers : 5.13.1

LOADING POOLED A2D2 TRAINING POOL

Rows: 2875

Columns:
   patient
   filename
   image_path
   class_folder
   label
   class_name
   global_serial
   global_serial_number
   window_serial
   window_serial_number

Image path column: image_path
Label column     : label

Class counts:
Seizure     : 575
Non-seizure : 2300
Total       : 2875

All 2875 image paths exist.

LOADING PATIENT 08 A2D2 BLINDED TEST

Rows: 1000

Columns:
   blinded_order
   blinded_id
   blinded_filename
   blinded_image_path
   global_serial
   true_label
   true_class

Image path column: blinded_image_path
Label column     : true_label

Class counts:
Seizure     : 200
Non-seizure : 800
Total       : 1000

All 1000 image paths exist.

LOADING PATIENT 11 A2D2 BLINDED TEST

Rows: 1000

C

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


SUCCESS: MedGemma-4B loaded.
GPU allocated: 8.02 GB
Gradient checkpointing disabled.
All pretrained MedGemma parameters frozen.


[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.



MEDGEMMA LANGUAGE STACK
Language layer path        : model.language_model.layers
Number of language layers  : 34
Language hidden size       : 2560
Intervention human layers  : [8, 17, 26, 31]
Intervention Python indices: [7, 16, 25, 30]
Number interventions       : 4

MULTI-LAYER LoReFT TRAINABLE PARAMETER REPORT
  base_model.model.language_model.layers.7._loreft_intervention.rotate_layer.parametrizations.weight.original: 10,240
  base_model.model.language_model.layers.7._loreft_intervention.learned_source.weight: 10,240
  base_model.model.language_model.layers.7._loreft_intervention.learned_source.bias: 4
  base_model.model.language_model.layers.16._loreft_intervention.rotate_layer.parametrizations.weight.original: 10,240
  base_model.model.language_model.layers.16._loreft_intervention.learned_source.weight: 10,240
  base_model.model.language_model.layers.16._loreft_intervention.learned_source.bias: 4
  base_model.model.language_model.layers.25._loreft_intervention.rotate_layer.param

... (9 outputs left out)

Patient 18 blinded test: 100%|██████████| 375/375 [02:40<00:00,  2.34it/s]



----------------------------------------------------------------------------------------------------
PATIENT 18 FINAL RESULTS
----------------------------------------------------------------------------------------------------
Test images         : 375
TN / FP / FN / TP   : 290 / 10 / 10 / 65
Sensitivity         : 0.8667
Specificity         : 0.9667
Balanced Accuracy   : 0.9167
Accuracy            : 0.9467
Precision           : 0.8667
F1                  : 0.8667
Invalid outputs     : 0
Predictions CSV     : C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_154258\blinded_tests\patient_18\patient_18_predictions.csv
Metrics JSON        : C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_154258\blinded_tests\patient_18\patient_18_metrics.json
Confusion CSV       : C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_154258\blinded_tests\patie

Patient 24 blinded test: 100%|██████████| 500/500 [03:35<00:00,  2.32it/s]



----------------------------------------------------------------------------------------------------
PATIENT 24 FINAL RESULTS
----------------------------------------------------------------------------------------------------
Test images         : 500
TN / FP / FN / TP   : 388 / 12 / 32 / 68
Sensitivity         : 0.6800
Specificity         : 0.9700
Balanced Accuracy   : 0.8250
Accuracy            : 0.9120
Precision           : 0.8500
F1                  : 0.7556
Invalid outputs     : 0
Predictions CSV     : C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_154258\blinded_tests\patient_24\patient_24_predictions.csv
Metrics JSON        : C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_154258\blinded_tests\patient_24\patient_24_metrics.json
Confusion CSV       : C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A2D2\LoReFT_A2D2_R4\261011_154258\blinded_tests\patie